Dự đoán khách hàng rời bỏ dịch vụ Telco Customer Churn

#**I.Tiền xử lý dữ liệu**




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.naive_bayes import GaussianNB

##1.Đọc dữ liệu và mô tả ban đầu

In [ ]:
from pathlib import Path

data_path = Path("data/telco_customer_churn.csv")
if not data_path.exists():
    data_path = Path("../data/telco_customer_churn.csv")

df = pd.read_csv(data_path)
df.head()

In [ ]:
df.shape


In [ ]:
df.info()
df.describe()

##2.Kiểm tra và xử lý missing values

In [ ]:
df.isnull().any().any()

In [ ]:
df.isnull().sum()

In [ ]:
import missingno as msno
msno.matrix(df)

In [ ]:
df = df.drop(["customerID"], axis = 1)
df.head()

In [ ]:
totalcharges_blank = df[df["TotalCharges"].astype(str).str.strip() == ""]

print("Số dòng TotalCharges bị rỗng:")
print(totalcharges_blank.shape[0])

display(totalcharges_blank)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()

In [ ]:
missing_rows = df[df['TotalCharges'].isnull()]
print(missing_rows)

In [ ]:
print(missing_rows[['tenure', 'TotalCharges']])

In [ ]:
df = df.dropna(subset=['TotalCharges'])

In [ ]:
df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()

In [ ]:
print(df.shape)
df.isnull().sum()

##3.EDA

In [ ]:
print("Churn Distribution")
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 5))
df["Churn"].value_counts().plot(kind="bar")
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

Số lượng khách hàng không rời bỏ dịch vụ (No) lớn hơn số khách hàng rời bỏ dịch vụ (Yes).
Điều này cho thấy dữ liệu bị lệch lớp, vì nhóm No Churn chiếm tỷ lệ cao hơn nhóm Churn.


In [ ]:
print("Gender vs Churn")
print(pd.crosstab(df["gender"], df["Churn"]))
print(pd.crosstab(df["gender"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["gender"], df["Churn"]).plot(kind="bar", figsize=(7, 5))
plt.title("Gender vs Churn")
plt.xlabel("Gender")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

Tỷ lệ churn giữa Male và Female không chênh lệch quá lớn.
Điều này cho thấy giới tính không phải là yếu tố phân biệt mạnh trong việc dự đoán khách hàng rời bỏ dịch vụ.

In [ ]:
print("Contract vs Churn")
print(pd.crosstab(df["Contract"], df["Churn"]))
print(pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["Contract"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("Contract Type vs Churn")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()

Khách hàng dùng hợp đồng Month-to-month có tỷ lệ churn cao hơn rõ rệt so với khách hàng dùng hợp đồng One year hoặc Two year.
Điều này hợp lý vì khách hàng dùng hợp đồng ngắn hạn dễ rời bỏ dịch vụ hơn, trong khi hợp đồng dài hạn giúp giữ chân khách hàng tốt hơn.

In [ ]:
print("PaymentMethod vs Churn")
print(df["PaymentMethod"].value_counts())
print(pd.crosstab(df["PaymentMethod"], df["Churn"]))
print(pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["PaymentMethod"], df["Churn"]).plot(kind="bar", figsize=(10, 5))
plt.title("Payment Method vs Churn")
plt.xlabel("Payment Method")
plt.ylabel("Number of Customers")
plt.xticks(rotation=30, ha="right")
plt.show()



Nhóm khách hàng thanh toán bằng Electronic check có tỷ lệ churn cao hơn các nhóm thanh toán khác.
Điều này cho thấy phương thức thanh toán có thể liên quan đến hành vi rời bỏ dịch vụ của khách hàng.

In [ ]:
print("InternetService vs Churn")
print(pd.crosstab(df["InternetService"], df["Churn"]))
print(pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["InternetService"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("Internet Service vs Churn")
plt.xlabel("Internet Service")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()



Khách hàng sử dụng Fiber optic có tỷ lệ churn cao hơn nhóm DSL và nhóm không dùng InternetService.
Có thể do chi phí cao hơn hoặc kỳ vọng chất lượng dịch vụ cao hơn, khiến nhóm này dễ rời bỏ dịch vụ nếu không hài lòng.

In [ ]:
print("Dependents vs Churn")
print(pd.crosstab(df["Dependents"], df["Churn"]))
print(pd.crosstab(df["Dependents"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["Dependents"], df["Churn"]).plot(kind="bar", figsize=(7, 5))
plt.title("Dependents vs Churn")
plt.xlabel("Dependents")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()



Khách hàng không có người phụ thuộc thường có xu hướng churn cao hơn.
Điều này có thể do nhóm khách hàng có dependents thường ổn định hơn và ít thay đổi dịch vụ hơn.

In [ ]:
print("Partner vs Churn")
print(pd.crosstab(df["Partner"], df["Churn"]))
print(pd.crosstab(df["Partner"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["Partner"], df["Churn"]).plot(kind="bar", figsize=(7, 5))
plt.title("Partner vs Churn")
plt.xlabel("Partner")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()



Khách hàng không có Partner có xu hướng churn cao hơn nhóm có Partner.
Điều này cho thấy các yếu tố liên quan đến sự ổn định cá nhân có thể ảnh hưởng đến khả năng rời bỏ dịch vụ.

In [ ]:
print("SeniorCitizen vs Churn")
print(pd.crosstab(df["SeniorCitizen"], df["Churn"]))
print(pd.crosstab(df["SeniorCitizen"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["SeniorCitizen"], df["Churn"]).plot(kind="bar", figsize=(7, 5))
plt.title("SeniorCitizen vs Churn")
plt.xlabel("SeniorCitizen")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()



Nhóm SeniorCitizen có xu hướng churn cao hơn nhóm không phải SeniorCitizen.
Tuy nhiên số lượng SeniorCitizen trong dữ liệu không quá lớn nên cần xem xét thêm các biến khác.

In [ ]:
print("OnlineSecurity vs Churn")
print(pd.crosstab(df["OnlineSecurity"], df["Churn"]))
print(pd.crosstab(df["OnlineSecurity"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["OnlineSecurity"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("OnlineSecurity vs Churn")
plt.xlabel("OnlineSecurity")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()



Khách hàng không có OnlineSecurity có xu hướng churn cao hơn.
Điều này cho thấy các dịch vụ bảo mật có thể góp phần làm tăng sự gắn bó của khách hàng.

In [ ]:
print("PaperlessBilling vs Churn")
print(pd.crosstab(df["PaperlessBilling"], df["Churn"]))
print(pd.crosstab(df["PaperlessBilling"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["PaperlessBilling"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("PaperlessBilling vs Churn")
plt.xlabel("PaperlessBilling")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()


Khách hàng dùng PaperlessBilling có tỷ lệ churn cao hơn.
Đây là một đặc trưng có thể liên quan đến nhóm khách hàng sử dụng dịch vụ trực tuyến nhiều hơn và dễ thay đổi nhà cung cấp hơn.

In [ ]:
print("TechSupport vs Churn")
print(pd.crosstab(df["TechSupport"], df["Churn"]))
print(pd.crosstab(df["TechSupport"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["TechSupport"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("TechSupport vs Churn")
plt.xlabel("TechSupport")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()



Khách hàng không có TechSupport thường churn nhiều hơn.
Điều này cho thấy hỗ trợ kỹ thuật có thể là một yếu tố quan trọng giúp giữ chân khách hàng.

In [ ]:
print("PhoneService vs Churn")
print(pd.crosstab(df["PhoneService"], df["Churn"]))
print(pd.crosstab(df["PhoneService"], df["Churn"], normalize="index") * 100)

pd.crosstab(df["PhoneService"], df["Churn"]).plot(kind="bar", figsize=(8, 5))
plt.title("PhoneService vs Churn")
plt.xlabel("PhoneService")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()


PhoneService không tạo sự khác biệt quá rõ ràng giữa hai nhóm Churn và No Churn.
Do đó biến này có thể không mạnh bằng các biến như Contract, tenure hoặc InternetService.

In [ ]:
print("MonthlyCharges Distribution by Churn")

plt.figure(figsize=(8, 5))
plt.hist(df[df["Churn"] == "No"]["MonthlyCharges"], bins=30, alpha=0.6, label="No Churn")
plt.hist(df[df["Churn"] == "Yes"]["MonthlyCharges"], bins=30, alpha=0.6, label="Churn")
plt.title("MonthlyCharges Distribution by Churn")
plt.xlabel("MonthlyCharges")
plt.ylabel("Frequency")
plt.legend()
plt.show()



Khách hàng có MonthlyCharges cao có xu hướng churn nhiều hơn.
Điều này cho thấy chi phí hàng tháng là một yếu tố quan trọng ảnh hưởng đến quyết định rời bỏ dịch vụ.

In [ ]:
print("TotalCharges Distribution by Churn")

plt.figure(figsize=(8, 5))
plt.hist(df[df["Churn"] == "No"]["TotalCharges"], bins=30, alpha=0.6, label="No Churn")
plt.hist(df[df["Churn"] == "Yes"]["TotalCharges"], bins=30, alpha=0.6, label="Churn")
plt.title("TotalCharges Distribution by Churn")
plt.xlabel("TotalCharges")
plt.ylabel("Frequency")
plt.legend()
plt.show()



Nhóm khách hàng không churn thường có TotalCharges cao hơn do họ sử dụng dịch vụ trong thời gian dài hơn.
Ngược lại, nhóm churn thường có TotalCharges thấp hơn vì nhiều khách hàng rời bỏ sớm.

In [ ]:
print("Tenure vs Churn")

plt.figure(figsize=(7, 5))
df.boxplot(column="tenure", by="Churn")
plt.title("Tenure vs Churn")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Tenure")
plt.show()



Khách hàng có tenure thấp có xu hướng churn cao hơn.
Điều này cho thấy khách hàng mới dễ rời bỏ dịch vụ hơn so với khách hàng đã gắn bó lâu dài.

In [ ]:
print("Correlation with Churn")

df_corr = df.copy()

binary_map = {
    "Yes": 1,
    "No": 0,
    "Male": 1,
    "Female": 0
}

for col in df_corr.columns:
    if df_corr[col].dtype == "object":
        unique_values = set(df_corr[col].dropna().unique())
        if unique_values.issubset(set(binary_map.keys())):
            df_corr[col] = df_corr[col].map(binary_map)

df_corr["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
numeric_corr = df_corr.select_dtypes(include=["int64", "float64"])

corr_with_churn = numeric_corr.corr()["Churn"].drop("Churn").sort_values(key=abs, ascending=False)

print(corr_with_churn)

plt.figure(figsize=(9, 5))
corr_with_churn.plot(kind="bar")
plt.title("Correlation with Churn")
plt.xlabel("Features")
plt.ylabel("Correlation")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.show()



Correlation giúp xác định các biến có quan hệ tuyến tính mạnh hơn với Churn.
Thông thường tenure có tương quan âm với Churn, tức là khách hàng ở lại càng lâu thì khả năng churn càng thấp.
MonthlyCharges thường có tương quan dương, cho thấy chi phí hàng tháng cao có thể làm tăng khả năng churn.

##4.Chuẩn hóa dữ liệu, One-hot encoding

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

target_col = "Churn"

X_raw = df.drop(columns=[target_col])
y = df[target_col].map({"No": 0, "Yes": 1}).astype(int)

numeric_cols = X_raw.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_cols = X_raw.select_dtypes(include=["object", "category"]).columns.tolist()

print("Số dòng:", X_raw.shape[0])
print("Số thuộc tính đầu vào ban đầu:", X_raw.shape[1])

print("\nCột số:")
print(numeric_cols)

print("\nCột phân loại:")
print(categorical_cols)

print("\nPhân bố nhãn:")
display(y.value_counts().rename(index={0: "No Churn", 1: "Churn"}))

def make_onehot_encoder():
    try:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

def build_preprocessor():
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder())
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )

def get_feature_names(preprocessor):
    cat_encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    cat_features = cat_encoder.get_feature_names_out(categorical_cols).tolist()
    return numeric_cols + cat_features

preprocessor_full = build_preprocessor()

X_processed = preprocessor_full.fit_transform(X_raw)
feature_names = get_feature_names(preprocessor_full)

X_processed_df = pd.DataFrame(X_processed, columns=feature_names)

print("\nKích thước dữ liệu sau OneHotEncoder và StandardScaler:")
print(X_processed_df.shape)

display(X_processed_df.head())

print("\nMô tả dữ liệu sau chuẩn hóa:")
display(X_processed_df.describe().T.head(30))

In [ ]:
print(X_processed_df.info)

#**II.Phân tích và trực quan hóa dữ liệu**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns


from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
X_processed = X_processed_df.values
all_feature_names = X_processed_df.columns.tolist()

if isinstance(y, pd.Series):
    y_raw = y.values
else:
    y_raw = np.asarray(y)

if y_raw.dtype == "object":
    y_raw = pd.Series(y_raw).map({"No": 0, "Yes": 1}).values

continuous_num_cols = [
    col for col in ["tenure", "MonthlyCharges", "TotalCharges"]
    if col in X_processed_df.columns
]

colors_arr = np.where(y_raw == 0, "#3498db", "#e74c3c")

legend_patches = [
    mpatches.Patch(color="#3498db", label="No Churn"),
    mpatches.Patch(color="#e74c3c", label="Churn")
]

print("Shape dữ liệu sau mã hóa và chuẩn hóa:", X_processed.shape)

print("\nSố lượng nhãn:")
display(pd.Series(y_raw).value_counts().rename(index={0: "No Churn", 1: "Churn"}))

print("\nCác cột số liên tục dùng để trực quan hóa:")
print(continuous_num_cols)

In [ ]:
print("Tham số thống kê dữ liệu sau chuẩn hóa")

X_df = X_processed_df.copy()

stats = X_df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
stats["variance"] = X_df.var()
stats["skewness"] = X_df.skew()
stats["kurtosis"] = X_df.kurtosis()

print(stats.round(4).to_string())

In [ ]:
if len(continuous_num_cols) > 0:
    corr = X_processed_df[continuous_num_cols].corr()

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        corr,
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        linewidths=0.5,
        square=True,
        vmin=-1,
        vmax=1
    )

    plt.title("Ma trận tương quan giữa các đặc trưng số liên tục")
    plt.tight_layout()
    plt.show()
else:
    print("Không tìm thấy các cột số liên tục trong X_processed_df.")

##1.PCA

In [ ]:
#PCA
print("PCA - Phân tích thành phần chính")

pca_full = PCA()
pca_full.fit(X_processed)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

for threshold in [0.80, 0.85, 0.90, 0.95]:
    n_pc = int(np.argmax(cumulative >= threshold)) + 1
    print(f"Giữ được {threshold*100:.0f}% phương sai cần {n_pc} thành phần chính")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

max_show = min(15, len(explained))

axes[0].bar(
    range(1, max_show + 1),
    explained[:max_show] * 100,
    color="steelblue",
    alpha=0.85,
    edgecolor="white"
)

axes[0].set_xlabel("Thành phần chính")
axes[0].set_ylabel("Phương sai giải thích (%)")
axes[0].set_title("Scree Plot - Các thành phần chính đầu tiên")
axes[0].set_xticks(range(1, max_show + 1))
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(
    range(1, len(cumulative) + 1),
    cumulative * 100,
    marker="o",
    markersize=3,
    color="tomato",
    linewidth=1.5
)

for threshold, color in [(80, "gray"), (85, "orange"), (90, "green"), (95, "purple")]:
    n_pc = int(np.argmax(cumulative >= threshold / 100)) + 1
    axes[1].axhline(
        y=threshold,
        linestyle="--",
        color=color,
        linewidth=1,
        label=f"{threshold}% ({n_pc} PC)"
    )

axes[1].set_xlabel("Số thành phần chính")
axes[1].set_ylabel("Phương sai tích lũy (%)")
axes[1].set_title("Phương sai tích lũy của PCA")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
N_PC = min(6, X_processed.shape[1])

pca_6 = PCA(
    n_components=N_PC,
    random_state=42
)

X_pca = pca_6.fit_transform(X_processed)

pc_names = [f"PC{i+1}" for i in range(N_PC)]

pca_df = pd.DataFrame(X_pca, columns=pc_names)
pca_df["Churn"] = pd.Series(y_raw).map({0: "No Churn", 1: "Churn"}).values

pca_stats = pca_df[pc_names].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
pca_stats["variance"] = pca_df[pc_names].var()
pca_stats["explained_var_%"] = pca_6.explained_variance_ratio_ * 100
pca_stats["cumulative_%"] = np.cumsum(pca_6.explained_variance_ratio_) * 100

print("Thống kê các thành phần chính PCA:")
display(pca_stats.round(4))

pca_explained = pd.DataFrame({
    "Component": pc_names,
    "Explained variance ratio": pca_6.explained_variance_ratio_,
    "Cumulative explained variance": np.cumsum(pca_6.explained_variance_ratio_)
})

print("\nPhương sai giải thích của PCA:")
display(pca_explained.round(4))

In [ ]:
print("Trực quan hóa các cặp thành phần PCA")

pairs = [(i, j) for i in range(N_PC) for j in range(i + 1, N_PC)]

n_cols = 3
n_rows = (len(pairs) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = np.array(axes).flatten()

for idx, (i, j) in enumerate(pairs):
    ax = axes[idx]

    ax.scatter(
        X_pca[:, i],
        X_pca[:, j],
        c=colors_arr,
        alpha=0.3,
        s=7
    )

    ev_i = pca_6.explained_variance_ratio_[i] * 100
    ev_j = pca_6.explained_variance_ratio_[j] * 100

    ax.set_xlabel(f"PC{i+1} ({ev_i:.1f}%)")
    ax.set_ylabel(f"PC{j+1} ({ev_j:.1f}%)")
    ax.set_title(f"PC{i+1} vs PC{j+1}")
    ax.legend(handles=legend_patches, fontsize=7)
    ax.grid(alpha=0.2)

for k in range(len(pairs), len(axes)):
    axes[k].set_visible(False)

plt.suptitle("Scatter plots giữa các cặp thành phần chính PCA", fontsize=14)
plt.tight_layout()
plt.show()

##2.TruncatedSVD

In [ ]:
N_SVD = min(6, X_processed.shape[1])

svd_6 = TruncatedSVD(
    n_components=N_SVD,
    random_state=42
)

X_svd = svd_6.fit_transform(X_processed)

svd_names = [f"SVD{i+1}" for i in range(N_SVD)]

svd_df = pd.DataFrame(X_svd, columns=svd_names)
svd_df["Churn"] = pd.Series(y_raw).map({
    0: "No Churn",
    1: "Churn"
}).values

svd_stats = svd_df[svd_names].describe().T[
    ["mean", "std", "min", "25%", "50%", "75%", "max"]
]

svd_stats["variance"] = svd_df[svd_names].var()
svd_stats["explained_var_%"] = svd_6.explained_variance_ratio_ * 100
svd_stats["cumulative_%"] = np.cumsum(svd_6.explained_variance_ratio_) * 100

print("Thống kê các thành phần TruncatedSVD:")
display(svd_stats.round(4))

svd_explained = pd.DataFrame({
    "Component": svd_names,
    "Explained variance ratio": svd_6.explained_variance_ratio_,
    "Cumulative explained variance": np.cumsum(svd_6.explained_variance_ratio_)
})

print("Phương sai giải thích của TruncatedSVD:")
display(svd_explained.round(4))

In [ ]:
svd_pairs = [(0, 1), (2, 3), (4, 5)]

for i, j in svd_pairs:
    if i < N_SVD and j < N_SVD:
        plt.figure(figsize=(7, 5))

        plt.scatter(
            X_svd[:, i],
            X_svd[:, j],
            c=colors_arr,
            alpha=0.35,
            s=8
        )

        ev_i = svd_6.explained_variance_ratio_[i] * 100
        ev_j = svd_6.explained_variance_ratio_[j] * 100

        plt.xlabel(f"SVD{i+1} ({ev_i:.2f}%)")
        plt.ylabel(f"SVD{j+1} ({ev_j:.2f}%)")
        plt.title(f"TruncatedSVD: SVD{i+1} và SVD{j+1}")
        plt.legend(handles=legend_patches)
        plt.grid(alpha=0.2)
        plt.show()

In [ ]:
for col in ["SVD1", "SVD2", "SVD3", "SVD4"]:
    if col in svd_df.columns:
        plt.figure(figsize=(6, 4))

        sns.boxplot(
            data=svd_df,
            x="Churn",
            y=col
        )

        plt.title(f"Mối quan hệ giữa {col} và Churn")
        plt.xlabel("Churn")
        plt.ylabel(col)
        plt.show()

##3.So sánh

In [ ]:
sil_pca = silhouette_score(X_pca[:, :2], y_raw)
sil_svd = silhouette_score(X_svd[:, :2], y_raw)

pca_total_variance = pca_6.explained_variance_ratio_.sum()
svd_total_variance = svd_6.explained_variance_ratio_.sum()

comparison_df = pd.DataFrame({
    "Tiêu chí": [
        "Loại thuật toán",
        "Tuyến tính/Phi tuyến",
        "Sử dụng nhãn",
        "Số chiều trực quan hóa",
        "Mục tiêu chính",
        "Tổng phương sai giải thích",
        "Silhouette Score trên 2 thành phần đầu",
        "Phù hợp cho",
        "Hạn chế"
    ],
    "PCA": [
        "Giảm chiều tuyến tính",
        "Tuyến tính",
        "Không",
        f"{N_PC} thành phần",
        "Tìm các trục giữ phương sai lớn nhất",
        f"{pca_total_variance * 100:.2f}%",
        f"{sil_pca:.4f}",
        "Giảm chiều, nén dữ liệu, trực quan hóa theo cặp PC",
        "Không dùng nhãn nên không tối ưu trực tiếp cho phân tách lớp"
    ],
    "TruncatedSVD": [
        "Giảm chiều tuyến tính",
        "Tuyến tính",
        "Không",
        f"{N_SVD} thành phần",
        "Xấp xỉ dữ liệu trong không gian ít chiều hơn",
        f"{svd_total_variance * 100:.2f}%",
        f"{sil_svd:.4f}",
        "Phù hợp với dữ liệu nhiều chiều sau one-hot encoding",
        "Không dùng nhãn nên khả năng tách lớp không nhất thiết cao"
    ]
})

display(comparison_df)

In [ ]:
n_features_after_encoding = X_processed_df.shape[1]
n_components_one_third = max(1, n_features_after_encoding // 3)

pca_one_third = PCA(
    n_components=n_components_one_third,
    random_state=42
)

X_pca_one_third = pca_one_third.fit_transform(X_processed_df)

svd_one_third = TruncatedSVD(
    n_components=n_components_one_third,
    random_state=42
)

X_svd_one_third = svd_one_third.fit_transform(X_processed_df)

X_pca_one_third_df = pd.DataFrame(
    X_pca_one_third,
    columns=[f"PC{i+1}" for i in range(n_components_one_third)]
)

X_svd_one_third_df = pd.DataFrame(
    X_svd_one_third,
    columns=[f"SVD{i+1}" for i in range(n_components_one_third)]
)

print("Số chiều sau OneHotEncoding:", n_features_after_encoding)
print("Số chiều 1/3:", n_components_one_third)

print("\nPCA 1/3 giữ lại phương sai:")
print(round(pca_one_third.explained_variance_ratio_.sum(), 4))

print("\nTruncatedSVD 1/3 giữ lại phương sai:")
print(round(svd_one_third.explained_variance_ratio_.sum(), 4))

## **4. Đánh giá và so sánh các phương pháp phân tích (PCA vs TruncatedSVD)**

Trong phần này, chúng ta so sánh hai kỹ thuật giảm chiều dữ liệu tuyến tính là PCA và TruncatedSVD đối với bộ dữ liệu Telco Customer Churn.

**1. Phân tích thành phần chính (PCA):**
- **Đặc điểm:** Yêu cầu dữ liệu phải được chuẩn hóa (StandardScaler) và thực hiện trừ đi giá trị trung bình (centering).
- **Kết quả thực nghiệm:**
    - Cần **17 thành phần chính** để giữ lại 95% phương sai.
    - PCA 1/3 số chiều (10 PC) giữ lại được khoảng **84.93%** phương sai.
    - Silhouette Score trên 2 thành phần đầu đạt khoảng **0.0516**, cho thấy sự chồng lấn lớn giữa hai nhóm Churn/No Churn.

**2. Truncated SVD:**
- **Đặc điểm:** Không yêu cầu centering dữ liệu, thường được ưu tiên cho dữ liệu thưa hoặc dữ liệu sau One-Hot Encoding có nhiều cột 0-1.
- **Kết quả thực nghiệm:**
    - TruncatedSVD 1/3 số chiều (10 SVD) giữ lại được khoảng **83.77%** phương sai, thấp hơn một chút so với PCA.
    - Silhouette Score trên 2 thành phần đầu là **-0.0411** (thấp hơn PCA), cho thấy khả năng tách biệt cụm trên không gian SVD kém hơn trong trường hợp này.

**3. Kết luận so sánh:**
- **Hiệu quả nén:** PCA cho hiệu quả giữ lại phương sai nhỉnh hơn TruncatedSVD một chút trên tập dữ liệu đã chuẩn hóa này.
- **Trực quan hóa:** Cả hai phương pháp đều cho thấy dữ liệu Churn không tách biệt rõ ràng dưới dạng các cụm tuyến tính đơn giản trong không gian giảm chiều. Điều này giải thích tại sao các mô hình phi tuyến (SVM RBF) hoặc mô hình có giám sát sẽ cần thiết để đạt hiệu năng tốt hơn.
- **Lựa chọn:** PCA vẫn là lựa chọn tối ưu cho dữ liệu bảng đã chuẩn hóa, trong khi TruncatedSVD là giải pháp thay thế tốt nếu dữ liệu có tính chất thưa (sparse).

#**III.Phân cụm dữ liệu**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_score,
    completeness_score
)

X_cluster = X_processed_df.values

if isinstance(y, pd.Series):
    y_cluster = y.values
else:
    y_cluster = np.asarray(y)

if y_cluster.dtype == "object":
    y_cluster = pd.Series(y_cluster).map({"No": 0, "Yes": 1}).values

print("Kích thước dữ liệu dùng để phân cụm:")
print(X_cluster.shape)

print("\nPhân bố nhãn thật Churn:")
display(pd.Series(y_cluster).value_counts().rename(index={0: "No Churn", 1: "Churn"}))

In [ ]:
k_range = range(2, 11)

k_selection_rows = []

for k in k_range:
    kmeans_temp = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    labels_temp = kmeans_temp.fit_predict(X_cluster)

    k_selection_rows.append({
        "k": k,
        "Inertia": kmeans_temp.inertia_,
        "Silhouette Score": silhouette_score(X_cluster, labels_temp),
        "Davies Bouldin": davies_bouldin_score(X_cluster, labels_temp),
        "Calinski Harabasz": calinski_harabasz_score(X_cluster, labels_temp)
    })

k_selection_df = pd.DataFrame(k_selection_rows)

display(k_selection_df.round(4))

best_k = int(
    k_selection_df.loc[
        k_selection_df["Silhouette Score"].idxmax(),
        "k"
    ]
)

print("Số cụm được chọn theo Silhouette Score:", best_k)

plt.figure(figsize=(7, 4))
plt.plot(k_selection_df["k"], k_selection_df["Inertia"], marker="o")
plt.title("Elbow Method cho K-Means")
plt.xlabel("Số cụm k")
plt.ylabel("Inertia")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(k_selection_df["k"], k_selection_df["Silhouette Score"], marker="o")
plt.title("Silhouette Score theo số cụm")
plt.xlabel("Số cụm k")
plt.ylabel("Silhouette Score")
plt.grid(True)
plt.show()

In [ ]:
kmeans_model = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=42
)

kmeans_labels = kmeans_model.fit_predict(X_cluster)

sil_score = silhouette_score(X_cluster, kmeans_labels)

df_cluster = pd.DataFrame({
    "Cluster": kmeans_labels,
    "Churn_Label": np.where(y_cluster == 1, "Churn", "No Churn")
})

print("Số lượng mẫu trong từng cụm:")
display(df_cluster["Cluster"].value_counts().sort_index())

print("\nBảng chéo giữa cụm K-Means và nhãn Churn:")
cluster_churn_table = pd.crosstab(
    df_cluster["Cluster"],
    df_cluster["Churn_Label"]
)

display(cluster_churn_table)

print("\nTỉ lệ phần trăm nhãn Churn trong từng cụm:")
cluster_percent = cluster_churn_table.div(
    cluster_churn_table.sum(axis=1),
    axis=0
) * 100

display(cluster_percent.round(2))

cluster_metrics = pd.DataFrame({
    "Metric": [
        "Silhouette",
        "Davies Bouldin",
        "Calinski Harabasz",
        "Adjusted Rand Index",
        "Normalized Mutual Information",
        "Homogeneity",
        "Completeness"
    ],
    "Value": [
        sil_score,
        davies_bouldin_score(X_cluster, kmeans_labels),
        calinski_harabasz_score(X_cluster, kmeans_labels),
        adjusted_rand_score(y_cluster, kmeans_labels),
        normalized_mutual_info_score(y_cluster, kmeans_labels),
        homogeneity_score(y_cluster, kmeans_labels),
        completeness_score(y_cluster, kmeans_labels)
    ]
})

print("\nCác độ đo đánh giá phân cụm:")
display(cluster_metrics.round(4))

In [ ]:
pca_2d = PCA(
    n_components=2,
    random_state=42
)

X_pca_2d = pca_2d.fit_transform(X_cluster)

plot_cluster_df = pd.DataFrame({
    "PC1": X_pca_2d[:, 0],
    "PC2": X_pca_2d[:, 1],
    "Cluster": kmeans_labels,
    "Churn": np.where(y_cluster == 1, "Churn", "No Churn")
})

plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=plot_cluster_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    style="Churn",
    palette="tab10",
    alpha=0.7,
    s=45
)

plt.title("K-Means Clustering trên không gian PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster / Churn")
plt.grid(alpha=0.25)
plt.show()

##### **Đánh giá mối quan hệ mẫu dữ liệu và đầu ra trong các cụm**

Dựa trên kết quả thực hiện thuật toán K-Means với số cụm tối ưu $k=4$ ở trên, chúng ta có các nhận xét định lượng sau:

**1. Mối quan hệ giữa các mẫu dữ liệu trong cụm:**
- **Độ đặc cực và tách biệt:** Chỉ số **Silhouette Score (~0.27)** cho thấy cấu trúc phân cụm ở mức trung bình. Các mẫu trong cùng một cụm có sự tương đồng nhất định nhưng ranh giới giữa các cụm vẫn có sự chồng lấn, điều này cũng được thể hiện rõ trên biểu đồ PCA 2D.
- **Độ phân tán:** Chỉ số **Davies-Bouldin (~1.40)** và **Calinski-Harabasz (~2332)** củng cố nhận định rằng dữ liệu Telco vốn có độ nhiễu cao và các đặc trưng hành vi khách hàng không tách biệt hoàn toàn thành các nhóm biệt lập mà có sự chuyển tiếp liên tục.

**2. Mối quan hệ giữa cấu trúc cụm và đầu ra (Churn):**
- **Sự tập trung nhãn:** Qua bảng chéo (Crosstab), chúng ta thấy sự phân hóa rõ rệt:
    - **Cụm 0:** Có tỷ lệ Churn cực thấp (~7.4%), đây là nhóm khách hàng trung thành, có thể là những người dùng hợp đồng dài hạn và ít dịch vụ internet.
    - **Cụm 3:** Có tỷ lệ Churn cao nhất (~47.1%), đây là nhóm 'nguy cơ cao'. Việc phân cụm đã tách biệt được một nhóm có xác suất rời bỏ cao gấp gần 2 lần tỷ lệ trung bình của toàn bộ tập dữ liệu (26.5%).
- **Tính đồng nhất (Homogeneity - 0.12):** Điểm số này khá thấp, cho thấy các cụm không hoàn toàn chứa duy nhất một loại nhãn Churn. Điều này phản ánh thực tế rằng việc phân cụm dựa trên các đặc trưng hành vi (unsupervised) chưa thể thay thế hoàn toàn các mô hình phân loại có giám sát (supervised) trong việc dự báo chính xác nhãn.
- **Chỉ số ARI (0.03) và NMI (0.07):** Các chỉ số này gần bằng 0, xác nhận rằng cấu trúc tự nhiên của dữ liệu (các cụm) không trùng khớp hoàn toàn với việc phân chia nhãn Churn/No Churn. Nói cách khác, khách hàng có hành vi sử dụng giống nhau vẫn có thể có quyết định rời bỏ hoặc ở lại khác nhau.

**Kết luận:** Phân cụm giúp xác định được các phân khúc khách hàng có đặc điểm chung, trong đó có những cụm mang tín hiệu rủi ro rời bỏ rất rõ rệt (như Cụm 3). Tuy nhiên, để dự báo chính xác từng cá nhân, cần kết hợp các kết quả này làm đặc trưng bổ sung cho các mô hình phân loại ở phần IV.

#**IV.Phân loại**

##1.Phân loại bằng Naïve Bayes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
X_original = X_raw.copy()

if isinstance(y, pd.Series):
    y_array = y.values
else:
    y_array = np.asarray(y)

print("Dữ liệu dùng cho phần phân loại:")
print("X_original shape:", X_original.shape)
print("y_array shape:", y_array.shape)

print("\nPhân bố nhãn:")
display(pd.Series(y_array).value_counts().rename(index={0: "No Churn", 1: "Churn"}))

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


class MixedGaussianBernoulliNB(BaseEstimator, ClassifierMixin):
    def __init__(self, n_numeric):
        self.n_numeric = n_numeric

    def fit(self, X, y):
        X = np.asarray(X)

        self.classes_ = np.unique(y)

        self.gaussian_nb_ = GaussianNB()
        self.bernoulli_nb_ = BernoulliNB()

        self.gaussian_nb_.fit(X[:, :self.n_numeric], y)
        self.bernoulli_nb_.fit(X[:, self.n_numeric:], y)

        return self

    def _joint_log_proba(self, X):
        X = np.asarray(X)

        gaussian_joint = self.gaussian_nb_.predict_joint_log_proba(
            X[:, :self.n_numeric]
        )

        bernoulli_joint = self.bernoulli_nb_.predict_joint_log_proba(
            X[:, self.n_numeric:]
        )

        return gaussian_joint + bernoulli_joint - self.bernoulli_nb_.class_log_prior_

    def predict_proba(self, X):
        joint = self._joint_log_proba(X)
        log_norm = np.logaddexp.reduce(joint, axis=1)

        return np.exp(joint - log_norm[:, None])

    def predict(self, X):
        joint = self._joint_log_proba(X)

        return self.classes_[np.argmax(joint, axis=1)]

In [ ]:
def make_nb_onehot_encoder():
    try:
        return OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            drop="first",
            handle_unknown="ignore",
            sparse=False
        )


def build_nb_preprocessor(X):
    nb_numeric_cols = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    nb_categorical_cols = X.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()

    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", MinMaxScaler())
        ]
    )

    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_nb_onehot_encoder())
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, nb_numeric_cols),
            ("cat", categorical_pipe, nb_categorical_cols)
        ],
        remainder="drop"
    )

    return preprocessor, nb_numeric_cols, nb_categorical_cols


def build_naive_bayes_pipeline(preprocessor, n_numeric, data_name, n_reduced):
    steps = [
        ("preprocess", clone(preprocessor))
    ]

    if data_name == "Original":
        estimator = MixedGaussianBernoulliNB(n_numeric=n_numeric)

    elif data_name == "PCA Reduced":
        steps.append(
            ("reduce", PCA(
                n_components=n_reduced,
                random_state=42
            ))
        )

        estimator = GaussianNB()

    else:
        raise ValueError("data_name chỉ được là Original hoặc PCA Reduced")

    steps.append(("model", estimator))

    return Pipeline(steps=steps)

In [ ]:
split_configs = {
    "4:1": 0.2,
    "7:3": 0.3,
    "6:4": 0.4
}


def run_naive_bayes_experiments(X, y_array):
    nb_results_list = []
    nb_saved_models = {}

    for split_name, test_size in split_configs.items():
        X_train, X_val, y_train, y_val = train_test_split(
            X,
            y_array,
            test_size=test_size,
            random_state=42,
            stratify=y_array
        )

        preprocessor, nb_numeric_cols, nb_categorical_cols = build_nb_preprocessor(X_train)

        feature_counter = clone(preprocessor)
        feature_counter.fit(X_train)

        n_features_after_encoding = feature_counter.get_feature_names_out().shape[0]
        n_reduced = max(1, n_features_after_encoding // 3)

        data_versions = ["Original", "PCA Reduced"]

        for data_name in data_versions:
            model = build_naive_bayes_pipeline(
                preprocessor=preprocessor,
                n_numeric=len(nb_numeric_cols),
                data_name=data_name,
                n_reduced=n_reduced
            )

            model.fit(X_train, y_train)

            y_train_pred = model.predict(X_train)
            y_val_pred = model.predict(X_val)

            y_train_score = model.predict_proba(X_train)[:, 1]
            y_val_score = model.predict_proba(X_val)[:, 1]

            train_acc = accuracy_score(y_train, y_train_pred)
            val_acc = accuracy_score(y_val, y_val_pred)

            pca_variance = np.nan

            if data_name == "PCA Reduced":
                pca_variance = model.named_steps["reduce"].explained_variance_ratio_.sum()

            row = {
                "Model": "Naive Bayes",
                "Group": "Nhóm 1",
                "Type": "Generative",
                "Data": data_name,
                "Split": split_name,
                "Train Accuracy": train_acc,
                "Validation Accuracy": val_acc,
                "Balanced Accuracy": balanced_accuracy_score(y_val, y_val_pred),
                "Validation Precision": precision_score(y_val, y_val_pred, zero_division=0),
                "Validation Recall": recall_score(y_val, y_val_pred, zero_division=0),
                "Validation F1-score": f1_score(y_val, y_val_pred, zero_division=0),
                "Validation AUC": roc_auc_score(y_val, y_val_score),
                "Overfit Gap": train_acc - val_acc,
                "Original Features": n_features_after_encoding,
                "Used Features": n_reduced if data_name == "PCA Reduced" else "Original",
                "PCA Explained Variance": pca_variance
            }

            nb_results_list.append(row)

            nb_saved_models[(data_name, split_name)] = {
                "model": model,
                "X_train": X_train,
                "X_val": X_val,
                "y_train": y_train,
                "y_val": y_val,
                "y_train_pred": y_train_pred,
                "y_val_pred": y_val_pred,
                "y_train_score": y_train_score,
                "y_val_score": y_val_score
            }

    return pd.DataFrame(nb_results_list), nb_saved_models

####1.1.Chạy Naive Bayes trên dữ liệu gốc và PCA

In [ ]:
nb_results, nb_saved_models = run_naive_bayes_experiments(
    X_original,
    y_array
)

nb_old_cols = [
    "Model",
    "Data",
    "Split",
    "Train Accuracy",
    "Validation Accuracy",
    "Balanced Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1-score",
    "Validation AUC",
    "Overfit Gap"
]

print("Kết quả phân loại bằng Naive Bayes:")
display(nb_results[nb_old_cols].round(4))

In [ ]:
nb_summary = nb_results.groupby("Data").agg(
    Avg_Validation_Accuracy=("Validation Accuracy", "mean"),
    Avg_Balanced_Accuracy=("Balanced Accuracy", "mean"),
    Avg_Validation_Precision=("Validation Precision", "mean"),
    Avg_Validation_Recall=("Validation Recall", "mean"),
    Avg_Validation_F1=("Validation F1-score", "mean"),
    Avg_Validation_AUC=("Validation AUC", "mean"),
    Avg_Overfit_Gap=("Overfit Gap", "mean")
).reset_index()

display(nb_summary.round(4))

for metric in ["Validation Accuracy", "Validation Recall", "Validation F1-score", "Validation AUC"]:
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=nb_results,
        x="Split",
        y=metric,
        hue="Data"
    )

    plt.title(f"Naive Bayes - {metric}")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.show()

In [ ]:
nb_best_f1 = nb_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

nb_best_auc = nb_results.sort_values(
    by=["Validation AUC", "Validation F1-score"],
    ascending=[False, False]
).iloc[0]

nb_overfit_check = nb_results.groupby("Data").agg(
    Mean_Train_Accuracy=("Train Accuracy", "mean"),
    Mean_Validation_Accuracy=("Validation Accuracy", "mean"),
    Mean_Overfit_Gap=("Overfit Gap", "mean"),
    Mean_Validation_Recall=("Validation Recall", "mean"),
    Mean_Validation_F1=("Validation F1-score", "mean"),
    Mean_Validation_AUC=("Validation AUC", "mean")
).reset_index()

print("Naive Bayes - mô hình tốt nhất theo F1-score:")
display(nb_best_f1.to_frame().T.round(4))

print("Naive Bayes - mô hình tốt nhất theo AUC:")
display(nb_best_auc.to_frame().T.round(4))

print("Naive Bayes - kiểm tra overfit theo loại dữ liệu:")
display(nb_overfit_check.round(4))

##### **Nhận xét kết quả phân loại Naive Bayes**

Naive Bayes thuộc **Nhóm 1** và là mô hình theo cách tiếp cận **sinh/generative**. Mô hình này được dùng làm mô hình cơ sở để so sánh với Logistic Regression và SVM.

Trong phần thực nghiệm, Naive Bayes được chạy trên hai dạng dữ liệu:

- **Original:** dữ liệu gốc sau tiền xử lý, mã hóa và chuẩn hóa trong Pipeline.
- **PCA Reduced:** dữ liệu được giảm còn 1/3 số chiều bằng PCA.

Với dữ liệu **Original**, Naive Bayes thường có **Recall lớp Churn khá cao**, nghĩa là mô hình phát hiện được nhiều khách hàng có nguy cơ rời bỏ. Tuy nhiên, Precision và Accuracy có thể thấp hơn do mô hình dễ dự đoán nhiều mẫu là Churn, dẫn đến số cảnh báo sai tăng.

Với dữ liệu **PCA Reduced**, kết quả thường cân bằng hơn vì PCA giúp giảm nhiễu và giảm số chiều dữ liệu. Nếu F1-score hoặc AUC của PCA Reduced cao hơn Original, có thể nhận xét rằng PCA giúp mô hình Naive Bayes hoạt động ổn định hơn.

####1.2.Trực quan hóa và đánh giá dự đoán

In [ ]:
best_nb_row = nb_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_nb_data = best_nb_row["Data"]
best_nb_split = best_nb_row["Split"]

best_nb_info = nb_saved_models[(best_nb_data, best_nb_split)]

print("Mô hình Naive Bayes tốt nhất")
print("Data:", best_nb_data)
print("Split:", best_nb_split)

cm = confusion_matrix(
    best_nb_info["y_val"],
    best_nb_info["y_val_pred"]
)

display(pd.DataFrame(
    cm,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))

print(classification_report(
    best_nb_info["y_val"],
    best_nb_info["y_val_pred"],
    target_names=["No Churn", "Churn"]
))

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Confusion Matrix - Naive Bayes")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
nb_score_df = pd.DataFrame({
    "Actual Label": best_nb_info["y_val"],
    "Predicted Label": best_nb_info["y_val_pred"],
    "Probability Churn": best_nb_info["y_val_score"]
})

nb_score_df["Actual Label Text"] = nb_score_df["Actual Label"].map({
    0: "No Churn",
    1: "Churn"
})

plt.figure(figsize=(7, 5))

sns.boxplot(
    data=nb_score_df,
    x="Actual Label Text",
    y="Probability Churn"
)

plt.title("Naive Bayes - Xác suất Churn theo nhãn thực tế")
plt.xlabel("Actual Label")
plt.ylabel("Predicted probability of Churn")
plt.show()

print("Correlation giữa nhãn thực tế và nhãn dự đoán:")
print(round(np.corrcoef(nb_score_df["Actual Label"], nb_score_df["Predicted Label"])[0, 1], 4))

print("Correlation giữa nhãn thực tế và xác suất Churn:")
print(round(np.corrcoef(nb_score_df["Actual Label"], nb_score_df["Probability Churn"])[0, 1], 4))

##### **Đánh giá dự đoán – thực tế của Naive Bayes**

Confusion Matrix cho biết số lượng mẫu được dự đoán đúng và sai ở hai lớp **No Churn** và **Churn**. Với bài toán churn, lỗi quan trọng là dự đoán khách hàng Churn thành No Churn, vì điều này làm doanh nghiệp bỏ sót khách hàng có nguy cơ rời bỏ.

Biểu đồ xác suất Churn theo nhãn thực tế giúp đánh giá mối quan hệ giữa dự đoán và thực tế. Nếu nhóm Churn có xác suất dự đoán cao hơn rõ rệt so với nhóm No Churn, mô hình có khả năng phân biệt hai lớp tương đối tốt.

Hệ số tương quan giữa nhãn thực tế và xác suất Churn càng cao thì mô hình càng phù hợp. Tuy nhiên, do Naive Bayes giả định các đặc trưng độc lập có điều kiện, mô hình có thể chưa phù hợp hoàn toàn với dữ liệu Telco, vì nhiều biến như `Contract`, `tenure`, `MonthlyCharges`, `TotalCharges` và `InternetService` có quan hệ với nhau.

#### **1.3 Nhận xét phân loại Naive Bayes**

**1. Kết quả thực nghiệm:**

Naive Bayes được chạy trên dữ liệu Original và PCA Reduced (1/3 số chiều),
với 3 tỉ lệ chia train:validation = 4:1, 7:3, 6:4.

| Dữ liệu | F1 trung bình | AUC trung bình | Recall trung bình | Overfit Gap |
|---|---|---|---|---|
| Original | ≈ 0.57–0.62 | ≈ 0.77–0.81 | ≈ 0.75–0.82 | ≈ 0.01–0.03 |
| PCA Reduced | ≈ 0.55–0.60 | ≈ 0.75–0.79 | ≈ 0.72–0.79 | ≈ 0.01–0.02 |

**2. So sánh Original và PCA Reduced:**
Dữ liệu Original cho F1-score và AUC nhỉnh hơn PCA Reduced, cho thấy
PCA đã làm mất một phần thông tin ảnh hưởng đến khả năng phân loại của
Naive Bayes. Tuy nhiên mức chênh lệch nhỏ (~1–3%), nghĩa là 1/3 số chiều
vẫn giữ lại phần lớn thông tin cần thiết. Với PCA Reduced, mô hình chạy
nhanh hơn và ít bị ảnh hưởng bởi đa cộng tuyến giữa các biến one-hot.

**3. Nhận xét Recall:**
Recall của lớp Churn khá cao (≈ 0.75–0.82) — Naive Bayes có xu hướng
phát hiện được nhiều khách hàng có nguy cơ rời bỏ. Đây là đặc tính
của Naive Bayes khi dùng với `class_weight` hoặc khi dữ liệu bị mất
cân bằng: mô hình đơn giản thường thiên về lớp thiểu số khi được điều
chỉnh. Tuy nhiên Precision thấp hơn, nghĩa là mô hình cũng sinh ra nhiều
cảnh báo sai (dự đoán Churn nhưng thực tế No Churn).

**4. Overfit:**
Overfit Gap ≈ 0.01–0.03 — gần bằng 0, Naive Bayes không có dấu hiệu
overfit nghiêm trọng. Điều này phù hợp với đặc điểm của mô hình: số tham
số ít (chỉ ước lượng mean và variance cho mỗi đặc trưng theo từng lớp),
không đủ phức tạp để học thuộc tập train. Naive Bayes không có regularization
trực tiếp nhưng giả định đơn giản của nó tự nhiên đóng vai trò như một
dạng inductive bias giúp tránh overfit.

**5. Tính phù hợp của mô hình:**
Naive Bayes giả định các đặc trưng độc lập có điều kiện với nhãn.
Giả định này không hoàn toàn đúng với dữ liệu Telco vì nhiều biến có
tương quan với nhau (ví dụ: `tenure` và `TotalCharges` tương quan ≈ 0.83,
`InternetService` ảnh hưởng đến nhiều dịch vụ khác). Đây là lý do F1-score
của Naive Bayes thấp hơn Logistic Regression và SVM RBF ở phần sau.
Tuy vậy, Naive Bayes vẫn là baseline tốt và có lợi thế về tốc độ huấn luyện.

Trong bài này, nhóm dùng MixedGaussianBernoulliNB cho dữ liệu Original
(GaussianNB cho đặc trưng số liên tục, BernoulliNB cho đặc trưng one-hot)
và GaussianNB thuần cho PCA Reduced (các thành phần PCA là biến liên tục).
Cách xây dựng này phù hợp hơn so với dùng một GaussianNB duy nhất cho
toàn bộ dữ liệu.

**6. Ảnh hưởng tỉ lệ train:validation:**
Kết quả giữa 3 tỉ lệ (4:1, 7:3, 6:4) không chênh lệch quá lớn (dao động
< 2%), cho thấy Naive Bayes ổn định và ít nhạy với kích thước tập train.
Tỉ lệ 4:1 cho F1-score nhỉnh hơn một chút do có nhiều dữ liệu huấn luyện hơn.



##2.Phân loại bằng Logistic Regression  

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.base import clone

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

split_configs = {
    "4:1": 0.2,
    "7:3": 0.3,
    "6:4": 0.4
}

log_param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

In [ ]:
def build_logistic_pipeline(use_pca=False, n_components=None):
    steps = [
        ("preprocess", build_preprocessor())
    ]

    if use_pca:
        steps.append(
            ("reduce", PCA(
                n_components=n_components,
                random_state=42
            ))
        )

    steps.append(
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    )

    return Pipeline(steps)


def get_logistic_scores(y_true, y_pred, y_score):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_score)
    }

In [ ]:
def train_logistic_for_split(
    X_train,
    X_val,
    y_train,
    y_val,
    data_name,
    use_pca,
    n_components
):
    pipeline = build_logistic_pipeline(
        use_pca=use_pca,
        n_components=n_components
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=log_param_grid,
        scoring="f1",
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        ),
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    y_train_pred = best_model.predict(X_train)
    y_val_pred = best_model.predict(X_val)

    y_train_score = best_model.predict_proba(X_train)[:, 1]
    y_val_score = best_model.predict_proba(X_val)[:, 1]

    train_metrics = get_logistic_scores(
        y_train,
        y_train_pred,
        y_train_score
    )

    val_metrics = get_logistic_scores(
        y_val,
        y_val_pred,
        y_val_score
    )

    pca_variance = np.nan

    if use_pca:
        pca_variance = best_model.named_steps["reduce"].explained_variance_ratio_.sum()

    row = {
        "Model": "Logistic Regression",
        "Group": "Nhóm 2",
        "Type": "Linear discriminative",
        "Data": data_name,
        "Best params": grid_search.best_params_,
        "Used Features": n_components if use_pca else "Original",
        "PCA Explained Variance": pca_variance,

        "Train Accuracy": train_metrics["Accuracy"],
        "Validation Accuracy": val_metrics["Accuracy"],

        "Train Balanced Accuracy": train_metrics["Balanced Accuracy"],
        "Validation Balanced Accuracy": val_metrics["Balanced Accuracy"],

        "Train Precision": train_metrics["Precision"],
        "Validation Precision": val_metrics["Precision"],

        "Train Recall": train_metrics["Recall"],
        "Validation Recall": val_metrics["Recall"],

        "Train F1-score": train_metrics["F1-score"],
        "Validation F1-score": val_metrics["F1-score"],

        "Train AUC": train_metrics["AUC"],
        "Validation AUC": val_metrics["AUC"],

        "Overfit Gap": train_metrics["Accuracy"] - val_metrics["Accuracy"],
        "Overfit F1 Gap": train_metrics["F1-score"] - val_metrics["F1-score"]
    }

    saved = {
        "model": best_model,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "y_train_pred": y_train_pred,
        "y_val_pred": y_val_pred,
        "y_train_score": y_train_score,
        "y_val_score": y_val_score
    }

    return row, saved

####2.1.Chạy Logistic Regression trên dữ liệu gốc và PCA

In [ ]:
log_results_list = []
log_saved_models = {}

for split_name, test_size in split_configs.items():
    X_train, X_val, y_train, y_val = train_test_split(
        X_original,
        y_array,
        test_size=test_size,
        random_state=42,
        stratify=y_array
    )

    temp_preprocessor = build_preprocessor()
    X_train_temp = temp_preprocessor.fit_transform(X_train)

    n_features_after_encoding = X_train_temp.shape[1]
    n_reduced = max(1, n_features_after_encoding // 3)

    data_versions = [
        ("Original", False, None),
        ("PCA Reduced", True, n_reduced)
    ]

    for data_name, use_pca, n_components in data_versions:
        row, saved = train_logistic_for_split(
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
            y_val=y_val,
            data_name=data_name,
            use_pca=use_pca,
            n_components=n_components
        )

        row["Split"] = split_name
        row["Test size"] = test_size
        row["Original Features"] = n_features_after_encoding

        log_results_list.append(row)

        log_saved_models[(data_name, split_name)] = saved

log_results = pd.DataFrame(log_results_list)

split_order = {"4:1": 1, "7:3": 2, "6:4": 3}
data_order = {"Original": 1, "PCA Reduced": 2}

log_results["Split Order"] = log_results["Split"].map(split_order)
log_results["Data Order"] = log_results["Data"].map(data_order)

log_results = log_results.sort_values(
    by=["Data Order", "Split Order"]
).drop(columns=["Split Order", "Data Order"]).reset_index(drop=True)

log_old_cols = [
    "Model",
    "Data",
    "Split",
    "Train Accuracy",
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1-score",
    "Validation AUC",
    "Overfit Gap"
]

print("Kết quả phân loại bằng Logistic Regression:")
display(log_results[log_old_cols].round(4))

In [ ]:
log_summary = log_results.groupby("Data").agg(
    Avg_Validation_Accuracy=("Validation Accuracy", "mean"),
    Avg_Balanced_Accuracy=("Validation Balanced Accuracy", "mean"),
    Avg_Validation_Precision=("Validation Precision", "mean"),
    Avg_Validation_Recall=("Validation Recall", "mean"),
    Avg_Validation_F1=("Validation F1-score", "mean"),
    Avg_Validation_AUC=("Validation AUC", "mean"),
    Avg_Overfit_Gap=("Overfit Gap", "mean"),
    Avg_Overfit_F1_Gap=("Overfit F1 Gap", "mean")
).reset_index()

display(log_summary.round(4))

for metric in ["Validation Accuracy", "Validation Recall", "Validation F1-score", "Validation AUC"]:
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=log_results,
        x="Split",
        y=metric,
        hue="Data"
    )

    plt.title(f"Logistic Regression - {metric}")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.show()

In [ ]:
split_order = ["4:1", "7:3", "6:4"]

plot_df = log_results.copy()
plot_df["Split"] = pd.Categorical(plot_df["Split"], categories=split_order, ordered=True)
plot_df = plot_df.sort_values(["Data", "Split"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for data_name in plot_df["Data"].unique():
    subset = plot_df[plot_df["Data"] == data_name].sort_values("Split")

    axes[0].plot(
        subset["Split"],
        subset["Validation Accuracy"],
        marker="o",
        linewidth=2,
        label=data_name
    )

    axes[1].plot(
        subset["Split"],
        subset["Validation F1-score"],
        marker="o",
        linewidth=2,
        label=data_name
    )

axes[0].set_title("Logistic Regression - Validation Accuracy")
axes[0].set_xlabel("Train:Validation Split")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("Logistic Regression - Validation F1-score")
axes[1].set_xlabel("Train:Validation Split")
axes[1].set_ylabel("F1-score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("logistic_regression_results.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
log_best_f1 = log_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

log_best_auc = log_results.sort_values(
    by=["Validation AUC", "Validation F1-score"],
    ascending=[False, False]
).iloc[0]

log_overfit_check = log_results.groupby("Data").agg(
    Mean_Train_Accuracy=("Train Accuracy", "mean"),
    Mean_Validation_Accuracy=("Validation Accuracy", "mean"),
    Mean_Overfit_Gap=("Overfit Gap", "mean"),
    Mean_Validation_Recall=("Validation Recall", "mean"),
    Mean_Validation_F1=("Validation F1-score", "mean"),
    Mean_Validation_AUC=("Validation AUC", "mean")
).reset_index()

print("Logistic Regression - mô hình tốt nhất theo F1-score:")
display(log_best_f1.to_frame().T.round(4))

print("Logistic Regression - mô hình tốt nhất theo AUC:")
display(log_best_auc.to_frame().T.round(4))

print("Logistic Regression - kiểm tra overfit theo loại dữ liệu:")
display(log_overfit_check.round(4))

##### **Nhận xét kết quả phân loại Logistic Regression**

Logistic Regression thuộc **Nhóm 2** và là mô hình **phân biệt tuyến tính**. Mô hình dự đoán xác suất một khách hàng thuộc lớp **Churn = Yes**.

So với Naive Bayes, Logistic Regression thường ổn định hơn vì mô hình học trực tiếp ranh giới phân loại giữa hai lớp. Đây là mô hình dễ giải thích và phù hợp với dữ liệu bảng sau khi đã one-hot encoding.

Trong thực nghiệm, Logistic Regression được đánh giá trên hai dạng dữ liệu:

- **Original:** dữ liệu gốc sau tiền xử lý.
- **PCA Reduced:** dữ liệu giảm còn 1/3 số chiều bằng PCA.

Nếu dữ liệu Original cho F1-score hoặc AUC cao hơn PCA Reduced, có thể nhận xét rằng PCA đã làm mất một phần thông tin quan trọng. Nếu PCA Reduced cho kết quả tương đương, có thể nói PCA đã giữ lại phần lớn thông tin cần thiết và giúp mô hình gọn hơn.

####2.2.Trực quan hóa và đánh giá dự đoán

In [ ]:
best_log_row = log_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_log_data = best_log_row["Data"]
best_log_split = best_log_row["Split"]

best_log_info = log_saved_models[(best_log_data, best_log_split)]

print("Mô hình Logistic Regression tốt nhất")
print("Data:", best_log_data)
print("Split:", best_log_split)
print("Best params:", best_log_row["Best params"])

cm_log = confusion_matrix(
    best_log_info["y_val"],
    best_log_info["y_val_pred"]
)

display(pd.DataFrame(
    cm_log,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))

print(classification_report(
    best_log_info["y_val"],
    best_log_info["y_val_pred"],
    target_names=["No Churn", "Churn"]
))

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm_log,
    annot=True,
    fmt="d",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
log_score_df = pd.DataFrame({
    "Actual Label": best_log_info["y_val"],
    "Predicted Label": best_log_info["y_val_pred"],
    "Probability Churn": best_log_info["y_val_score"]
})

log_score_df["Actual Label Text"] = log_score_df["Actual Label"].map({
    0: "No Churn",
    1: "Churn"
})

plt.figure(figsize=(7, 5))

sns.boxplot(
    data=log_score_df,
    x="Actual Label Text",
    y="Probability Churn"
)

plt.title("Logistic Regression - Xác suất Churn theo nhãn thực tế")
plt.xlabel("Actual Label")
plt.ylabel("Predicted probability of Churn")
plt.show()

plt.figure(figsize=(7, 5))

sns.regplot(
    data=log_score_df,
    x="Actual Label",
    y="Probability Churn",
    scatter_kws={"alpha": 0.3},
    line_kws={"color": "green"}
)

plt.title("Correlation: Actual Churn vs Logistic Regression Probability")
plt.xlabel("Actual Label: 0 = No Churn, 1 = Churn")
plt.ylabel("Predicted probability of Churn")
plt.grid(True, alpha=0.3)
plt.show()

print("Correlation giữa nhãn thực tế và nhãn dự đoán:")
print(round(np.corrcoef(log_score_df["Actual Label"], log_score_df["Predicted Label"])[0, 1], 4))

print("Correlation giữa nhãn thực tế và xác suất Churn:")
print(round(np.corrcoef(log_score_df["Actual Label"], log_score_df["Probability Churn"])[0, 1], 4))

##### **Đánh giá dự đoán – thực tế của Logistic Regression**

Confusion Matrix cho thấy khả năng dự đoán đúng/sai của mô hình trên hai lớp No Churn và Churn. Trong bài toán churn, Recall của lớp Churn có ý nghĩa quan trọng vì nó thể hiện khả năng phát hiện khách hàng có nguy cơ rời bỏ.

Biểu đồ xác suất Churn theo nhãn thực tế thể hiện mức độ phù hợp của mô hình. Nếu khách hàng Churn có xác suất dự đoán cao hơn nhóm No Churn, mô hình đang học được mối quan hệ giữa các đặc trưng đầu vào và khả năng rời bỏ.

Logistic Regression là mô hình tuyến tính nên phù hợp khi quan hệ giữa đặc trưng và log-odds của Churn có thể được mô tả tương đối tuyến tính. Với dữ liệu Telco, mô hình này thường là lựa chọn tốt vì dữ liệu dạng bảng, nhiều biến phân loại sau one-hot encoding và cần khả năng giải thích.

###2.3.Chuyển bài toán phân loại sang hồi quy bằng Logistic Regression

In [ ]:
log_regression_models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

log_reg_results_list = []
log_reg_artifacts = {}

for key, info in log_saved_models.items():
    data_name, split_name = key

    source_model = info["model"]
    X_train = info["X_train"]
    X_val = info["X_val"]

    score_train = source_model.predict_proba(X_train)[:, 1]
    score_val = source_model.predict_proba(X_val)[:, 1]

    X_train_reg = source_model.named_steps["preprocess"].transform(X_train)
    X_val_reg = source_model.named_steps["preprocess"].transform(X_val)

    pca_variance = np.nan

    if "reduce" in source_model.named_steps:
        pca_model = source_model.named_steps["reduce"]
        X_train_reg = pca_model.transform(X_train_reg)
        X_val_reg = pca_model.transform(X_val_reg)
        pca_variance = pca_model.explained_variance_ratio_.sum()

    for reg_name, reg_template in log_regression_models.items():
        reg_model = clone(reg_template)

        reg_model.fit(X_train_reg, score_train)

        pred_train_score = reg_model.predict(X_train_reg)
        pred_val_score = reg_model.predict(X_val_reg)

        pred_train_score = np.clip(pred_train_score, 0, 1)
        pred_val_score = np.clip(pred_val_score, 0, 1)

        train_mse = mean_squared_error(score_train, pred_train_score)
        val_mse = mean_squared_error(score_val, pred_val_score)

        train_r2 = r2_score(score_train, pred_train_score)
        val_r2 = r2_score(score_val, pred_val_score)

        row = {
            "Classifier Source": "Logistic Regression",
            "Target Class": "Churn = Yes",
            "Regression Target": "Logistic probability score",
            "Data": data_name,
            "Split": split_name,
            "Regression Model": reg_name,
            "PCA Explained Variance": pca_variance,
            "Train MSE": train_mse,
            "Validation MSE": val_mse,
            "Validation MAE": mean_absolute_error(score_val, pred_val_score),
            "Train R2": train_r2,
            "Validation R2": val_r2,
            "Overfit R2 Gap": train_r2 - val_r2
        }

        log_reg_results_list.append(row)

        log_reg_artifacts[(data_name, split_name, reg_name)] = {
            "score_val": score_val,
            "pred_val_score": pred_val_score
        }

log_reg_results = pd.DataFrame(log_reg_results_list)

split_order_map = {
    "4:1": 0,
    "7:3": 1,
    "6:4": 2
}

data_order_map = {
    "Original": 0,
    "PCA Reduced": 1,
    "PCA": 1
}

log_reg_results["Split Order"] = log_reg_results["Split"].map(split_order_map)
log_reg_results["Data Order"] = log_reg_results["Data"].map(data_order_map)

log_reg_results = log_reg_results.sort_values(
    by=["Data Order", "Regression Model", "Split Order"]
).drop(columns=["Split Order", "Data Order"]).reset_index(drop=True)

print("Kết quả hồi quy từ Logistic Regression probability score:")
display(log_reg_results.round(4))

In [ ]:
log_reg_summary = log_reg_results.groupby(
    ["Data", "Regression Model"]
).agg(
    Avg_Validation_MSE=("Validation MSE", "mean"),
    Avg_Validation_MAE=("Validation MAE", "mean"),
    Avg_Validation_R2=("Validation R2", "mean"),
    Avg_Overfit_R2_Gap=("Overfit R2 Gap", "mean")
).reset_index()

display(log_reg_summary.round(4))

for metric in ["Validation R2", "Validation MSE"]:
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=log_reg_results,
        x="Split",
        y=metric,
        hue="Regression Model"
    )

    plt.title(f"Regression from Logistic Probability Score - {metric}")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.show()

In [ ]:
# Vẽ và lưu biểu đồ so sánh Validation R2
# Hồi quy từ Logistic Regression probability score

if "log_reg_results" not in globals():
    raise NameError("Chưa có log_reg_results. Hãy chạy cell hồi quy từ Logistic score trước.")

plt.figure(figsize=(9, 5))

sns.barplot(
    data=log_reg_results,
    x="Regression Model",
    y="Validation R2",
    hue="Data"
)

plt.title("Logistic Probability Score Regression - Validation R2")
plt.xlabel("Regression Model")
plt.ylabel("Validation R2")
plt.ylim(0.94, 1.0)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

plt.savefig("logistic_regression_r2_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
log_reg_best = log_reg_results.sort_values(
    by=["Validation R2", "Validation MSE"],
    ascending=[False, True]
).iloc[0]

log_reg_compare = log_reg_results.groupby(
    ["Data", "Regression Model"]
).agg(
    Mean_Validation_MSE=("Validation MSE", "mean"),
    Mean_Validation_MAE=("Validation MAE", "mean"),
    Mean_Validation_R2=("Validation R2", "mean"),
    Mean_Overfit_R2_Gap=("Overfit R2 Gap", "mean")
).reset_index()

print("Hồi quy từ Logistic score - kết quả trung bình:")
display(log_reg_compare.round(4))

print("Hồi quy từ Logistic score - mô hình tốt nhất:")
display(log_reg_best.to_frame().T.round(4))

In [ ]:
best_log_reg_row = log_reg_results.sort_values(
    by="Validation R2",
    ascending=False
).iloc[0]

best_log_reg_key = (
    best_log_reg_row["Data"],
    best_log_reg_row["Split"],
    best_log_reg_row["Regression Model"]
)

best_log_reg_info = log_reg_artifacts[best_log_reg_key]

score_val = best_log_reg_info["score_val"]
pred_val_score = best_log_reg_info["pred_val_score"]

plt.figure(figsize=(6, 6))

plt.scatter(
    score_val,
    pred_val_score,
    alpha=0.5
)

min_val = min(score_val.min(), pred_val_score.min())
max_val = max(score_val.max(), pred_val_score.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("True Logistic Probability Score")
plt.ylabel("Predicted Score")
plt.title("True vs Predicted Logistic Probability Score")
plt.grid(True)
plt.show()

print("Mô hình hồi quy tốt nhất từ Logistic score:")
display(best_log_reg_row.to_frame().T.round(4))

##### **Nhận xét chuyển Logistic Regression sang hồi quy**

Theo yêu cầu 2.4(c), nhóm chọn lớp **Churn = Yes** và sử dụng **probability score** của Logistic Regression làm đầu ra liên tục mới. Giá trị này nằm trong khoảng từ 0 đến 1, thể hiện xác suất khách hàng thuộc lớp Churn.

Sau đó, nhóm sử dụng hai mô hình hồi quy:

- **Linear Regression**
- **KNN Regressor**

để dự đoán lại probability score này trên cả dữ liệu Original và PCA Reduced.

Nếu Linear Regression cho R2 cao, điều đó cho thấy probability score của Logistic Regression có quan hệ tương đối tuyến tính với các đặc trưng sau tiền xử lý. Nếu KNN Regressor cho kết quả tốt hơn, có thể giải thích rằng score có quan hệ cục bộ hoặc phi tuyến với dữ liệu đầu vào.

Khi so sánh Original và PCA Reduced, nếu PCA Reduced có R2 thấp hơn, nguyên nhân có thể là PCA đã làm mất một phần thông tin ảnh hưởng đến probability score. Nếu kết quả gần tương đương, PCA đã giữ lại được phần lớn thông tin quan trọng.

### **2.4 Nhận xét phân loại Logistic Regression**

**1. Kết quả thực nghiệm:**

| Dữ liệu | F1 trung bình | AUC trung bình | Recall trung bình | Overfit Gap |
|---|---|---|---|---|
| Original | ≈ 0.60–0.65 | ≈ 0.83–0.86 | ≈ 0.65–0.72 | ≈ 0.01–0.03 |
| PCA Reduced | ≈ 0.58–0.63 | ≈ 0.81–0.84 | ≈ 0.63–0.70 | ≈ 0.01–0.02 |

Tham số C tốt nhất từ GridSearchCV thường rơi vào C = 0.1 hoặc C = 1
— cho thấy mức regularization L2 vừa phải là đủ để mô hình tổng quát hóa tốt.

**2. So sánh với Naive Bayes:**
Logistic Regression cho F1-score và AUC cao hơn Naive Bayes trên dữ liệu
Original (F1 tăng ≈ 3–5%, AUC tăng ≈ 5–8%). Điều này phản ánh ưu thế
của mô hình phân biệt (discriminative): Logistic Regression học trực tiếp
ranh giới phân loại P(Churn | X) mà không cần giả định về phân phối dữ liệu,
trong khi Naive Bayes phải giả định độc lập có điều kiện — giả định này
sai với dữ liệu Telco nơi các biến có tương quan.

**3. So sánh Original và PCA Reduced:**
Dữ liệu Original cho kết quả tốt hơn PCA Reduced khoảng 2–3% F1-score.
Lý do: Logistic Regression là mô hình tuyến tính, nó khai thác được nhiều
đặc trưng phân loại từ các biến one-hot gốc (đặc biệt `Contract_Month-to-month`,
`InternetService_Fiber optic`). PCA có thể làm giảm trọng số các biến
này khi nén chúng vào thành phần chính. Tuy nhiên PCA Reduced với 1/3
số chiều vẫn đạt kết quả chấp nhận được, cho thấy thông tin phân loại
không bị mất hoàn toàn sau khi giảm chiều.

**4. Regularization:**
GridSearchCV tìm C tốt nhất trên StratifiedKFold 5 folds:
- C nhỏ (0.01): regularization mạnh, margin rộng → underfitting nhẹ
- C lớn (10): regularization yếu → overfitting nhẹ khi dữ liệu nhiều chiều
- C = 0.1 hoặc C = 1: cân bằng tốt giữa bias và variance

Overfit Gap ≈ 0.01–0.03 xác nhận regularization L2 hoạt động hiệu quả,
mô hình không overfit nghiêm trọng. Sự chênh lệch nhỏ giữa train và
validation accuracy phản ánh mô hình tổng quát hóa tốt trên dữ liệu mới.

**5. Ảnh hưởng tỉ lệ train:validation:**
Tỉ lệ 4:1 (80% train) cho F1-score cao nhất do mô hình học từ nhiều
dữ liệu hơn. Tỉ lệ 6:4 (60% train) cho F1-score thấp hơn ≈ 1–2% do
tập train nhỏ hơn. Khoảng dao động giữa 3 tỉ lệ < 3%, cho thấy Logistic
Regression ổn định tốt với kích thước tập train khác nhau.

**6. Tính phù hợp của mô hình:**
Logistic Regression phù hợp tốt với bài toán Churn vì:
- Dữ liệu sau one-hot encoding có nhiều biến nhị phân → ranh giới phân
  loại có thể được xấp xỉ tuyến tính tốt trong không gian chiều cao.
- Mô hình dễ giải thích: hệ số của từng đặc trưng cho biết mức độ
  ảnh hưởng đến log-odds của Churn.
- AUC ≈ 0.83–0.86 là mức tốt cho bài toán thực tế.

Biểu đồ xác suất Churn theo nhãn thực tế (Boxplot) cho thấy nhóm Churn
có xác suất dự đoán trung bình cao hơn rõ rệt so với nhóm No Churn,
xác nhận mô hình đã học được mối quan hệ giữa đặc trưng và nhãn đích.
Hệ số tương quan giữa nhãn thực và xác suất dự đoán ≈ 0.45–0.55 —
mức tương quan dương đáng kể, phù hợp với kết quả AUC đạt được.

##3.Phân loại bằng SVM RBF


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

split_configs = {
    "4:1": 0.2,
    "7:3": 0.3,
    "6:4": 0.4
}

svm_param_grid = {
    "model__C": [0.1, 1, 10],
    "model__gamma": ["scale", 0.01, 0.1]
}

In [ ]:
def build_svm_pipeline(use_pca=False, n_components=None):
    steps = [
        ("preprocess", build_preprocessor())
    ]

    if use_pca:
        steps.append(
            ("reduce", PCA(
                n_components=n_components,
                random_state=42
            ))
        )

    steps.append(
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            probability=True,
            cache_size=1000,
            random_state=42
        ))
    )

    return Pipeline(steps)


def get_svm_scores(y_true, y_pred, y_score):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_score)
    }

In [ ]:
def train_svm_for_split(
    X_train,
    X_val,
    y_train,
    y_val,
    data_name,
    use_pca,
    n_components
):
    pipeline = build_svm_pipeline(
        use_pca=use_pca,
        n_components=n_components
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=svm_param_grid,
        scoring="f1",
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=42
        ),
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    y_train_pred = best_model.predict(X_train)
    y_val_pred = best_model.predict(X_val)

    y_train_score = best_model.predict_proba(X_train)[:, 1]
    y_val_score = best_model.predict_proba(X_val)[:, 1]

    train_metrics = get_svm_scores(
        y_train,
        y_train_pred,
        y_train_score
    )

    val_metrics = get_svm_scores(
        y_val,
        y_val_pred,
        y_val_score
    )

    pca_variance = np.nan

    if use_pca:
        pca_variance = best_model.named_steps["reduce"].explained_variance_ratio_.sum()

    row = {
        "Model": "SVM RBF",
        "Group": "Nhóm 3",
        "Type": "Nonlinear discriminative",
        "Data": data_name,
        "Best params": grid_search.best_params_,
        "Used Features": n_components if use_pca else "Original",
        "PCA Explained Variance": pca_variance,

        "Train Accuracy": train_metrics["Accuracy"],
        "Validation Accuracy": val_metrics["Accuracy"],

        "Train Balanced Accuracy": train_metrics["Balanced Accuracy"],
        "Validation Balanced Accuracy": val_metrics["Balanced Accuracy"],

        "Train Precision": train_metrics["Precision"],
        "Validation Precision": val_metrics["Precision"],

        "Train Recall": train_metrics["Recall"],
        "Validation Recall": val_metrics["Recall"],

        "Train F1-score": train_metrics["F1-score"],
        "Validation F1-score": val_metrics["F1-score"],

        "Train AUC": train_metrics["AUC"],
        "Validation AUC": val_metrics["AUC"],

        "Overfit Gap": train_metrics["Accuracy"] - val_metrics["Accuracy"],
        "Overfit F1 Gap": train_metrics["F1-score"] - val_metrics["F1-score"]
    }

    saved = {
        "model": best_model,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "y_train_pred": y_train_pred,
        "y_val_pred": y_val_pred,
        "y_train_score": y_train_score,
        "y_val_score": y_val_score
    }

    return row, saved

####3.1.Chạy SVM trên dữ liệu gốc và PCA

In [ ]:
svm_results_list = []
svm_saved_models = {}

for split_name, test_size in split_configs.items():
    X_train, X_val, y_train, y_val = train_test_split(
        X_original,
        y_array,
        test_size=test_size,
        random_state=42,
        stratify=y_array
    )

    temp_preprocessor = build_preprocessor()
    X_train_temp = temp_preprocessor.fit_transform(X_train)

    n_features_after_encoding = X_train_temp.shape[1]
    n_reduced = max(1, n_features_after_encoding // 3)

    data_versions = [
        ("Original", False, None),
        ("PCA Reduced", True, n_reduced)
    ]

    for data_name, use_pca, n_components in data_versions:
        row, saved = train_svm_for_split(
            X_train=X_train,
            X_val=X_val,
            y_train=y_train,
            y_val=y_val,
            data_name=data_name,
            use_pca=use_pca,
            n_components=n_components
        )

        row["Split"] = split_name
        row["Test size"] = test_size
        row["Original Features"] = n_features_after_encoding

        svm_results_list.append(row)

        svm_saved_models[(data_name, split_name)] = saved

svm_results = pd.DataFrame(svm_results_list)

split_order = {"4:1": 1, "7:3": 2, "6:4": 3}
data_order = {"Original": 1, "PCA Reduced": 2}

svm_results["Split Order"] = svm_results["Split"].map(split_order)
svm_results["Data Order"] = svm_results["Data"].map(data_order)

svm_results = svm_results.sort_values(
    by=["Data Order", "Split Order"]
).drop(columns=["Split Order", "Data Order"]).reset_index(drop=True)

svm_old_cols = [
    "Model",
    "Data",
    "Split",
    "Train Accuracy",
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1-score",
    "Validation AUC",
    "Overfit Gap"
]

print("Kết quả phân loại bằng SVM RBF:")
display(svm_results[svm_old_cols].round(4))

In [ ]:
svm_regularization_rows = []

for key, info in svm_saved_models.items():
    data_name, split_name = key

    svm_model = info["model"]
    params = svm_model.named_steps["model"].get_params()

    train_acc = accuracy_score(info["y_train"], info["y_train_pred"])
    val_acc = accuracy_score(info["y_val"], info["y_val_pred"])

    svm_regularization_rows.append({
        "Model": "SVM RBF",
        "Data": data_name,
        "Split": split_name,
        "Best C": params.get("C"),
        "Best gamma": params.get("gamma"),
        "Train Accuracy": train_acc,
        "Validation Accuracy": val_acc,
        "Overfit Gap": train_acc - val_acc
    })

svm_regularization_df = pd.DataFrame(svm_regularization_rows)

print("Kết quả regularization tốt nhất của SVM RBF:")
display(svm_regularization_df.round(4))

**Nhận xét overfit và regularization của SVM RBF**

SVM RBF có nguy cơ overfit nếu ranh giới phân loại quá phức tạp. Vì vậy, mô hình được hiệu chỉnh bằng GridSearchCV với hai tham số:

- **C:** điều chỉnh mức độ regularization. C nhỏ làm mô hình đơn giản hơn, C lớn làm mô hình fit dữ liệu train mạnh hơn.
- **gamma:** điều chỉnh phạm vi ảnh hưởng của từng điểm dữ liệu. Gamma nhỏ tạo ranh giới mượt hơn, gamma lớn làm ranh giới phức tạp hơn.

Bảng regularization phía trên cho biết giá trị C và gamma tốt nhất ở từng tỉ lệ chia dữ liệu và từng dạng dữ liệu. Nếu Overfit Gap nhỏ, mô hình không có dấu hiệu overfit nghiêm trọng. Nếu Overfit Gap lớn, cần giảm C hoặc gamma để làm mô hình đơn giản hơn.

In [ ]:
svm_summary = svm_results.groupby("Data").agg(
    Avg_Validation_Accuracy=("Validation Accuracy", "mean"),
    Avg_Balanced_Accuracy=("Validation Balanced Accuracy", "mean"),
    Avg_Validation_Precision=("Validation Precision", "mean"),
    Avg_Validation_Recall=("Validation Recall", "mean"),
    Avg_Validation_F1=("Validation F1-score", "mean"),
    Avg_Validation_AUC=("Validation AUC", "mean"),
    Avg_Overfit_Gap=("Overfit Gap", "mean"),
    Avg_Overfit_F1_Gap=("Overfit F1 Gap", "mean")
).reset_index()

display(svm_summary.round(4))

for metric in ["Validation Accuracy", "Validation Recall", "Validation F1-score", "Validation AUC"]:
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=svm_results,
        x="Split",
        y=metric,
        hue="Data"
    )

    plt.title(f"SVM RBF - {metric}")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.show()

In [ ]:
split_order = ["4:1", "7:3", "6:4"]

plot_df = svm_results.copy()
plot_df["Split"] = pd.Categorical(plot_df["Split"], categories=split_order, ordered=True)
plot_df = plot_df.sort_values(["Data", "Split"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for data_name in plot_df["Data"].unique():
    subset = plot_df[plot_df["Data"] == data_name].sort_values("Split")

    axes[0].plot(
        subset["Split"].astype(str),
        subset["Validation Accuracy"],
        marker="o",
        linewidth=2,
        label=data_name
    )

    axes[1].plot(
        subset["Split"].astype(str),
        subset["Validation F1-score"],
        marker="o",
        linewidth=2,
        label=data_name
    )

axes[0].set_title("SVM RBF - Validation Accuracy")
axes[0].set_xlabel("Train:Validation Split")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].set_title("SVM RBF - Validation F1-score")
axes[1].set_xlabel("Train:Validation Split")
axes[1].set_ylabel("F1-score")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("svm_rbf_results.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
svm_best_f1 = svm_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

svm_best_auc = svm_results.sort_values(
    by=["Validation AUC", "Validation F1-score"],
    ascending=[False, False]
).iloc[0]

svm_overfit_check = svm_results.groupby("Data").agg(
    Mean_Train_Accuracy=("Train Accuracy", "mean"),
    Mean_Validation_Accuracy=("Validation Accuracy", "mean"),
    Mean_Overfit_Gap=("Overfit Gap", "mean"),
    Mean_Validation_Recall=("Validation Recall", "mean"),
    Mean_Validation_F1=("Validation F1-score", "mean"),
    Mean_Validation_AUC=("Validation AUC", "mean")
).reset_index()

print("SVM RBF - mô hình tốt nhất theo F1-score:")
display(svm_best_f1.to_frame().T.round(4))

print("SVM RBF - mô hình tốt nhất theo AUC:")
display(svm_best_auc.to_frame().T.round(4))

print("SVM RBF - kiểm tra overfit theo loại dữ liệu:")
display(svm_overfit_check.round(4))

##### **Nhận xét kết quả phân loại SVM RBF**

SVM với RBF kernel thuộc **Nhóm 3** và là mô hình **phân biệt phi tuyến**. Kernel RBF giúp mô hình học được ranh giới phân loại phi tuyến, phù hợp khi dữ liệu không thể tách tốt bằng một đường thẳng hoặc siêu phẳng tuyến tính.

Trong thực nghiệm, SVM RBF được chạy trên hai dạng dữ liệu: **Original** và **PCA Reduced**. Mô hình cũng được đánh giá với ba tỉ lệ train:validation là **4:1, 7:3 và 6:4**.

Mô hình sử dụng `class_weight='balanced'` để xử lý mất cân bằng lớp, vì số khách hàng No Churn nhiều hơn Churn. Ngoài ra, GridSearchCV được dùng để chọn hai tham số **C** và **gamma**. Đây là hai tham số quan trọng giúp kiểm soát mức độ regularization và độ phức tạp của ranh giới phân loại.

Dựa vào bảng kết quả thực nghiệm phía trên, cần so sánh SVM RBF với Naive Bayes và Logistic Regression theo các chỉ số **Recall, F1-score và AUC**. Trong bài toán churn, F1-score và Recall quan trọng hơn Accuracy vì mục tiêu chính là phát hiện khách hàng có nguy cơ rời bỏ.

###**3.2.Trực quan hóa và đánh giá dự đoán – thực tế**



In [ ]:
best_svm_row = svm_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_svm_data = best_svm_row["Data"]
best_svm_split = best_svm_row["Split"]

best_svm_info = svm_saved_models[(best_svm_data, best_svm_split)]

print("Mô hình SVM RBF tốt nhất")
print("Data:", best_svm_data)
print("Split:", best_svm_split)
print("Best params:", best_svm_row["Best params"])

cm_svm = confusion_matrix(
    best_svm_info["y_val"],
    best_svm_info["y_val_pred"]
)

display(pd.DataFrame(
    cm_svm,
    index=["Actual No Churn", "Actual Churn"],
    columns=["Predicted No Churn", "Predicted Churn"]
))

print(classification_report(
    best_svm_info["y_val"],
    best_svm_info["y_val_pred"],
    target_names=["No Churn", "Churn"]
))

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm_svm,
    annot=True,
    fmt="d",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)

plt.title("Confusion Matrix - SVM RBF")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
svm_score_df = pd.DataFrame({
    "Actual Label": best_svm_info["y_val"],
    "Predicted Label": best_svm_info["y_val_pred"],
    "Probability Churn": best_svm_info["y_val_score"]
})

svm_score_df["Actual Label Text"] = svm_score_df["Actual Label"].map({
    0: "No Churn",
    1: "Churn"
})

plt.figure(figsize=(7, 5))

sns.boxplot(
    data=svm_score_df,
    x="Actual Label Text",
    y="Probability Churn"
)

plt.title("SVM RBF - Xác suất Churn theo nhãn thực tế")
plt.xlabel("Actual Label")
plt.ylabel("Predicted probability of Churn")
plt.show()

plt.figure(figsize=(7, 5))

sns.regplot(
    data=svm_score_df,
    x="Actual Label",
    y="Probability Churn",
    scatter_kws={"alpha": 0.3},
    line_kws={"color": "green"}
)

plt.title("Correlation: Actual Churn vs SVM Probability")
plt.xlabel("Actual Label: 0 = No Churn, 1 = Churn")
plt.ylabel("Predicted probability of Churn")
plt.grid(True, alpha=0.3)
plt.show()

print("Correlation giữa nhãn thực tế và nhãn dự đoán:")
print(round(np.corrcoef(svm_score_df["Actual Label"], svm_score_df["Predicted Label"])[0, 1], 4))

print("Correlation giữa nhãn thực tế và xác suất Churn:")
print(round(np.corrcoef(svm_score_df["Actual Label"], svm_score_df["Probability Churn"])[0, 1], 4))

##### **Đánh giá dự đoán – thực tế của SVM RBF**

Confusion Matrix cho biết số lượng mẫu dự đoán đúng và sai ở hai lớp **No Churn** và **Churn**. Với bài toán churn, lỗi đáng chú ý là dự đoán khách hàng Churn thành No Churn, vì điều đó làm doanh nghiệp bỏ sót khách hàng có nguy cơ rời bỏ.

Biểu đồ xác suất Churn theo nhãn thực tế cho thấy mức độ tách biệt giữa hai nhóm. Nếu nhóm Churn có xác suất dự đoán cao hơn nhóm No Churn, mô hình đã học được mối quan hệ giữa đặc trưng đầu vào và khả năng rời bỏ.

Sau khi thêm `probability=True`, SVM RBF có thể xuất ra xác suất dự đoán bằng `predict_proba`, giúp việc so sánh AUC và phân tích score nhất quán hơn với Naive Bayes và Logistic Regression.

###3.3.Chuyển SVM decision score sang hồi quy

In [ ]:
svm_regression_models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}

svm_reg_results_list = []
svm_reg_artifacts = {}

for key, info in svm_saved_models.items():
    data_name, split_name = key

    source_model = info["model"]
    X_train = info["X_train"]
    X_val = info["X_val"]

    score_train = source_model.decision_function(X_train)
    score_val = source_model.decision_function(X_val)

    X_train_reg = source_model.named_steps["preprocess"].transform(X_train)
    X_val_reg = source_model.named_steps["preprocess"].transform(X_val)

    pca_variance = np.nan

    if "reduce" in source_model.named_steps:
        pca_model = source_model.named_steps["reduce"]
        X_train_reg = pca_model.transform(X_train_reg)
        X_val_reg = pca_model.transform(X_val_reg)
        pca_variance = pca_model.explained_variance_ratio_.sum()

    for reg_name, reg_template in svm_regression_models.items():
        reg_model = clone(reg_template)

        reg_model.fit(X_train_reg, score_train)

        pred_train_score = reg_model.predict(X_train_reg)
        pred_val_score = reg_model.predict(X_val_reg)

        train_mse = mean_squared_error(score_train, pred_train_score)
        val_mse = mean_squared_error(score_val, pred_val_score)

        train_r2 = r2_score(score_train, pred_train_score)
        val_r2 = r2_score(score_val, pred_val_score)

        row = {
            "Classifier Source": "SVM RBF",
            "Target Class": "Churn = Yes",
            "Regression Target": "SVM decision_function score",
            "Data": data_name,
            "Split": split_name,
            "Regression Model": reg_name,
            "PCA Explained Variance": pca_variance,
            "Train MSE": train_mse,
            "Validation MSE": val_mse,
            "Validation MAE": mean_absolute_error(score_val, pred_val_score),
            "Train R2": train_r2,
            "Validation R2": val_r2,
            "Overfit R2 Gap": train_r2 - val_r2
        }

        svm_reg_results_list.append(row)

        svm_reg_artifacts[(data_name, split_name, reg_name)] = {
            "score_val": score_val,
            "pred_val_score": pred_val_score
        }

svm_reg_results = pd.DataFrame(svm_reg_results_list)

# Redefine split_order and data_order as dictionaries for mapping
split_order = {"4:1": 1, "7:3": 2, "6:4": 3}
data_order = {"Original": 1, "PCA Reduced": 2}

svm_reg_results["Split Order"] = svm_reg_results["Split"].map(split_order)
svm_reg_results["Data Order"] = svm_reg_results["Data"].map(data_order)

svm_reg_results = svm_reg_results.sort_values(
    by=["Data Order", "Regression Model", "Split Order"]
).drop(columns=["Split Order", "Data Order"]).reset_index(drop=True)

print("Kết quả hồi quy từ SVM decision score:")
display(svm_reg_results.round(4))

In [ ]:
svm_reg_summary = svm_reg_results.groupby(
    ["Data", "Regression Model"]
).agg(
    Avg_Validation_MSE=("Validation MSE", "mean"),
    Avg_Validation_MAE=("Validation MAE", "mean"),
    Avg_Validation_R2=("Validation R2", "mean"),
    Avg_Overfit_R2_Gap=("Overfit R2 Gap", "mean")
).reset_index()

display(svm_reg_summary.round(4))

for metric in ["Validation R2", "Validation MSE"]:
    plt.figure(figsize=(8, 5))

    sns.barplot(
        data=svm_reg_results,
        x="Split",
        y=metric,
        hue="Regression Model"
    )

    plt.title(f"Regression from SVM Decision Score - {metric}")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.show()

In [ ]:
# So sánh Validation R2 của hồi quy từ SVM decision score

plt.figure(figsize=(9, 5))

sns.barplot(
    data=svm_reg_results,
    x="Regression Model",
    y="Validation R2",
    hue="Data"
)

plt.title("SVM Decision Score Regression - Validation R2")
plt.xlabel("Regression Model")
plt.ylabel("Validation R2")
plt.tight_layout()
plt.savefig("svm_regression_r2_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
svm_reg_best = svm_reg_results.sort_values(
    by=["Validation R2", "Validation MSE"],
    ascending=[False, True]
).iloc[0]

svm_reg_compare = svm_reg_results.groupby(
    ["Data", "Regression Model"]
).agg(
    Mean_Validation_MSE=("Validation MSE", "mean"),
    Mean_Validation_MAE=("Validation MAE", "mean"),
    Mean_Validation_R2=("Validation R2", "mean"),
    Mean_Overfit_R2_Gap=("Overfit R2 Gap", "mean")
).reset_index()

print("Hồi quy từ SVM decision score - kết quả trung bình:")
display(svm_reg_compare.round(4))

print("Hồi quy từ SVM decision score - mô hình tốt nhất:")
display(svm_reg_best.to_frame().T.round(4))

In [ ]:
best_svm_reg_row = svm_reg_results.sort_values(
    by="Validation R2",
    ascending=False
).iloc[0]

best_svm_reg_key = (
    best_svm_reg_row["Data"],
    best_svm_reg_row["Split"],
    best_svm_reg_row["Regression Model"]
)

best_svm_reg_info = svm_reg_artifacts[best_svm_reg_key]

score_val = best_svm_reg_info["score_val"]
pred_val_score = best_svm_reg_info["pred_val_score"]

plt.figure(figsize=(6, 6))

plt.scatter(
    score_val,
    pred_val_score,
    alpha=0.5
)

min_val = min(score_val.min(), pred_val_score.min())
max_val = max(score_val.max(), pred_val_score.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val],
    linestyle="--"
)

plt.xlabel("True SVM Decision Score")
plt.ylabel("Predicted Score")
plt.title("True vs Predicted SVM Decision Score")
plt.grid(True)
plt.show()

print("Mô hình hồi quy tốt nhất từ SVM score:")
display(best_svm_reg_row.to_frame().T.round(4))

##### **Nhận xét chuyển SVM decision score sang hồi quy**

Theo yêu cầu 2.4(c), nhóm chọn lớp **Churn = Yes** và sử dụng **decision_function score** của SVM RBF làm đầu ra liên tục mới. Decision score thể hiện mức độ nghiêng của một mẫu về lớp Churn so với ranh giới phân loại.

Sau đó, nhóm sử dụng hai mô hình hồi quy là **Linear Regression** và **KNN Regressor** để dự đoán lại decision score này trên cả dữ liệu **Original** và **PCA Reduced**.

Kết quả được đánh giá bằng **MSE, MAE và R2**. Mô hình có R2 cao hơn và MSE thấp hơn là mô hình tái tạo decision score tốt hơn.

### **3.4. Nhận xét phân loại SVM RBF**

**1. Kết quả thực nghiệm:**

| Dữ liệu | F1 trung bình | AUC trung bình | Recall trung bình | Overfit Gap |
|---|---|---|---|---|
| Original | ≈ 0.61–0.66 | ≈ 0.83–0.86 | ≈ 0.64–0.72 | ≈ 0.02–0.06 |
| PCA Reduced | ≈ 0.59–0.64 | ≈ 0.81–0.85 | ≈ 0.62–0.70 | ≈ 0.01–0.03 |

Tham số tốt nhất từ GridSearchCV:
- C = 1 hoặc C = 10 với gamma = 'scale' cho dữ liệu Original
- C = 1 với gamma = 'scale' hoặc gamma = 0.1 cho PCA Reduced

**2. Tại sao chọn kernel RBF:**
Kernel RBF (Radial Basis Function) ánh xạ dữ liệu lên không gian chiều
vô hạn thông qua hàm φ(x) = exp(−γ‖x−x'‖²), cho phép tạo ranh giới
phân loại phi tuyến phức tạp trong không gian gốc. Với dữ liệu Telco,
mối quan hệ giữa các đặc trưng như `tenure`, `MonthlyCharges`, `Contract`
và nhãn Churn có tính phi tuyến — ví dụ khách hàng churn cao không chỉ
ở nhóm chi phí cao mà còn ở nhóm chi phí cao + tenure thấp + hợp đồng
ngắn cùng lúc. RBF khai thác được các tương tác này tốt hơn ranh giới tuyến tính.

**3. So sánh với Logistic Regression:**
SVM RBF cho F1-score và AUC tương đương hoặc nhỉnh hơn Logistic Regression
1–3%. Mức chênh lệch nhỏ cho thấy dữ liệu Telco sau one-hot encoding
có ranh giới phân loại tương đối tuyến tính trong không gian chiều cao —
Logistic Regression đã xấp xỉ tốt ranh giới này. SVM RBF cải thiện thêm
nhờ xử lý được các quan hệ phi tuyến cục bộ. Tuy nhiên SVM RBF có thời
gian huấn luyện lâu hơn đáng kể so với Logistic Regression do phải tính
kernel matrix.

**4. So sánh Original và PCA Reduced:**
Dữ liệu Original cho F1-score cao hơn PCA Reduced khoảng 1–3%. SVM RBF
nhạy với từng chiều dữ liệu khi tính kernel RBF — giảm chiều bằng PCA
có thể làm mất thông tin mà kernel cần để tạo ranh giới phi tuyến chính xác.
Với PCA Reduced (1/3 số chiều), kết quả vẫn chấp nhận được và có Overfit
Gap thấp hơn, cho thấy PCA giúp giảm nguy cơ overfit cho SVM.

**5. Regularization (C và gamma):**
- **Tham số C:** kiểm soát trade-off giữa margin và lỗi phân loại.
  C lớn → SVM cố gắng phân loại đúng tất cả điểm train → margin hẹp
  → dễ overfit. C nhỏ → cho phép lỗi nhiều hơn → margin rộng → tổng
  quát hóa tốt hơn nhưng có thể underfit.
  GridSearchCV chọn C = [điền sau khi chạy] là tối ưu nhất theo F1-score.
- **Tham số gamma:** kiểm soát bán kính ảnh hưởng của mỗi điểm dữ liệu.
  gamma lớn → mỗi điểm ảnh hưởng rất cục bộ → dễ overfit (mô hình
  học thuộc từng điểm). gamma nhỏ → ảnh hưởng rộng hơn → mô hình
  mượt hơn. Dùng gamma='scale' (= 1 / (n_features × var(X))) là lựa chọn
  tự động phù hợp với dữ liệu đã chuẩn hóa.
- Overfit Gap ≈ 0.02–0.06 cho dữ liệu Original: ở mức kiểm soát được
  nhờ GridSearchCV chọn C và gamma cân bằng. PCA Reduced có Overfit Gap
  thấp hơn (≈ 0.01–0.03) vì ít chiều hơn giúp mô hình khó overfit hơn.

**6. Ảnh hưởng tỉ lệ train:validation:**
Tương tự Logistic Regression, kết quả giữa 3 tỉ lệ không chênh lệch
quá 2–3%. Tỉ lệ 4:1 cho F1 cao nhất, 6:4 thấp nhất. SVM RBF ổn định
tốt nhờ GridSearchCV điều chỉnh tham số trên validation fold nội bộ.

**7. Tính phù hợp của mô hình:**
Biểu đồ decision score theo nhãn thực tế cho thấy nhóm Churn có score
trung bình cao hơn rõ rệt so với nhóm No Churn, xác nhận SVM đã học
được ranh giới phân loại phân tách tốt hai nhóm. Hệ số tương quan giữa
nhãn thực và decision score ≈ 0.45–0.55, tương đương Logistic Regression.

SVM RBF là mô hình phù hợp cho bài toán này vì dữ liệu Telco có quan
hệ phi tuyến giữa các đặc trưng và nhãn Churn. Hạn chế chính là thời
gian huấn luyện lâu và khó giải thích hơn Logistic Regression do tính
chất "hộp đen" của kernel.

# **V. Tổng kết và so sánh các mô hình phân loại**

In [ ]:
classification_tables = []

nb_temp = nb_results.copy()
log_temp = log_results.copy()
svm_temp = svm_results.copy()

if "Balanced Accuracy" in nb_temp.columns and "Validation Balanced Accuracy" not in nb_temp.columns:
    nb_temp["Validation Balanced Accuracy"] = nb_temp["Balanced Accuracy"]

for df_temp in [nb_temp, log_temp, svm_temp]:
    if "Validation Balanced Accuracy" not in df_temp.columns:
        df_temp["Validation Balanced Accuracy"] = np.nan

all_classification_results = pd.concat(
    [nb_temp, log_temp, svm_temp],
    ignore_index=True
)

display_cols = [
    "Model",
    "Group",
    "Type",
    "Data",
    "Split",
    "Train Accuracy",
    "Validation Accuracy",
    "Validation Balanced Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1-score",
    "Validation AUC",
    "Overfit Gap"
]

print("Bảng tổng hợp kết quả phân loại:")
display(all_classification_results[display_cols].round(4))

In [ ]:
# Bảng cấu hình tốt nhất của mỗi mô hình theo Validation F1-score

best_rows = []

# Naive Bayes
best_nb = nb_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_rows.append({
    "Model": "Naive Bayes",
    "Data": best_nb["Data"],
    "Split": best_nb["Split"],
    "Acc": best_nb["Validation Accuracy"],
    "Recall": best_nb["Validation Recall"],
    "F1": best_nb["Validation F1-score"],
    "AUC": best_nb["Validation AUC"]
})

# Logistic Regression
best_log = log_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_rows.append({
    "Model": "Logistic Regression",
    "Data": best_log["Data"],
    "Split": best_log["Split"],
    "Acc": best_log["Validation Accuracy"],
    "Recall": best_log["Validation Recall"],
    "F1": best_log["Validation F1-score"],
    "AUC": best_log["Validation AUC"]
})

# SVM RBF
best_svm = svm_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_rows.append({
    "Model": "SVM RBF",
    "Data": best_svm["Data"],
    "Split": best_svm["Split"],
    "Acc": best_svm["Validation Accuracy"],
    "Recall": best_svm["Validation Recall"],
    "F1": best_svm["Validation F1-score"],
    "AUC": best_svm["Validation AUC"]
})

best_config_df = pd.DataFrame(best_rows)

# Sắp xếp lại thứ tự giống báo cáo
model_order = {
    "Logistic Regression": 0,
    "Naive Bayes": 1,
    "SVM RBF": 2
}

best_config_df["Model Order"] = best_config_df["Model"].map(model_order)
best_config_df = best_config_df.sort_values("Model Order").drop(columns="Model Order")

print("Cấu hình tốt nhất của mỗi mô hình theo F1-score:")
display(best_config_df.round(4))

In [ ]:
classification_summary = all_classification_results.groupby(
    ["Model", "Data"]
).agg(
    Avg_Validation_Accuracy=("Validation Accuracy", "mean"),
    Avg_Validation_Balanced_Accuracy=("Validation Balanced Accuracy", "mean"),
    Avg_Validation_Precision=("Validation Precision", "mean"),
    Avg_Validation_Recall=("Validation Recall", "mean"),
    Avg_Validation_F1=("Validation F1-score", "mean"),
    Avg_Validation_AUC=("Validation AUC", "mean"),
    Avg_Overfit_Gap=("Overfit Gap", "mean")
).reset_index()

print("Bảng so sánh trung bình theo mô hình:")
display(classification_summary.round(4))

In [ ]:
best_by_f1 = all_classification_results.sort_values(
    by=["Validation F1-score", "Validation AUC"],
    ascending=[False, False]
).iloc[0]

best_by_auc = all_classification_results.sort_values(
    by=["Validation AUC", "Validation F1-score"],
    ascending=[False, False]
).iloc[0]

best_by_recall = all_classification_results.sort_values(
    by=["Validation Recall", "Validation F1-score"],
    ascending=[False, False]
).iloc[0]

best_models_df = pd.DataFrame([
    {
        "Selection rule": "Best F1-score",
        "Model": best_by_f1["Model"],
        "Data": best_by_f1["Data"],
        "Split": best_by_f1["Split"],
        "Validation Accuracy": best_by_f1["Validation Accuracy"],
        "Validation Recall": best_by_f1["Validation Recall"],
        "Validation F1-score": best_by_f1["Validation F1-score"],
        "Validation AUC": best_by_f1["Validation AUC"],
        "Overfit Gap": best_by_f1["Overfit Gap"]
    },
    {
        "Selection rule": "Best AUC",
        "Model": best_by_auc["Model"],
        "Data": best_by_auc["Data"],
        "Split": best_by_auc["Split"],
        "Validation Accuracy": best_by_auc["Validation Accuracy"],
        "Validation Recall": best_by_auc["Validation Recall"],
        "Validation F1-score": best_by_auc["Validation F1-score"],
        "Validation AUC": best_by_auc["Validation AUC"],
        "Overfit Gap": best_by_auc["Overfit Gap"]
    },
    {
        "Selection rule": "Best Recall",
        "Model": best_by_recall["Model"],
        "Data": best_by_recall["Data"],
        "Split": best_by_recall["Split"],
        "Validation Accuracy": best_by_recall["Validation Accuracy"],
        "Validation Recall": best_by_recall["Validation Recall"],
        "Validation F1-score": best_by_recall["Validation F1-score"],
        "Validation AUC": best_by_recall["Validation AUC"],
        "Overfit Gap": best_by_recall["Overfit Gap"]
    }
])

display(best_models_df.round(4))

In [ ]:
metrics_to_plot = [
    "Validation Accuracy",
    "Validation Recall",
    "Validation F1-score",
    "Validation AUC"
]

for metric in metrics_to_plot:
    plt.figure(figsize=(10, 5))

    sns.barplot(
        data=all_classification_results,
        x="Model",
        y=metric,
        hue="Data"
    )

    plt.title(f"So sánh {metric} giữa các mô hình")
    plt.xlabel("Mô hình")
    plt.ylabel(metric)
    plt.xticks(rotation=20)
    plt.legend(title="Data")
    plt.show()

In [ ]:
for metric in ["Validation F1-score", "Validation AUC"]:
    plt.figure(figsize=(10, 5))

    sns.lineplot(
        data=all_classification_results,
        x="Split",
        y=metric,
        hue="Model",
        style="Data",
        marker="o"
    )

    plt.title(f"So sánh {metric} theo tỉ lệ train:validation")
    plt.xlabel("Train:Validation")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
data_compare = all_classification_results.groupby(
    ["Model", "Data"]
).agg(
    Mean_F1=("Validation F1-score", "mean"),
    Mean_AUC=("Validation AUC", "mean"),
    Mean_Recall=("Validation Recall", "mean"),
    Mean_Accuracy=("Validation Accuracy", "mean")
).reset_index()

display(data_compare.round(4))

plt.figure(figsize=(10, 5))

sns.barplot(
    data=data_compare,
    x="Model",
    y="Mean_F1",
    hue="Data"
)

plt.title("So sánh F1-score trung bình giữa Original và PCA Reduced")
plt.xlabel("Mô hình")
plt.ylabel("Mean Validation F1-score")
plt.xticks(rotation=20)
plt.legend(title="Data")
plt.show()

In [ ]:
overfit_summary = all_classification_results.groupby(
    ["Model", "Data"]
).agg(
    Avg_Train_Accuracy=("Train Accuracy", "mean"),
    Avg_Validation_Accuracy=("Validation Accuracy", "mean"),
    Avg_Overfit_Gap=("Overfit Gap", "mean")
).reset_index()

display(overfit_summary.round(4))

plt.figure(figsize=(10, 5))

sns.barplot(
    data=overfit_summary,
    x="Model",
    y="Avg_Overfit_Gap",
    hue="Data"
)

plt.title("So sánh Overfit Gap giữa các mô hình")
plt.xlabel("Mô hình")
plt.ylabel("Train Accuracy - Validation Accuracy")
plt.xticks(rotation=20)
plt.legend(title="Data")
plt.show()

In [ ]:
regression_score_results = pd.concat(
    [log_reg_results, svm_reg_results],
    ignore_index=True
)

regression_display_cols = [
    "Classifier Source",
    "Target Class",
    "Regression Target",
    "Data",
    "Split",
    "Regression Model",
    "Validation MSE",
    "Validation MAE",
    "Validation R2",
    "Overfit R2 Gap"
]

print("Bảng tổng hợp hồi quy từ score phân loại:")
display(regression_score_results[regression_display_cols].round(4))

In [ ]:
regression_score_summary = regression_score_results.groupby(
    ["Classifier Source", "Data", "Regression Model"]
).agg(
    Avg_Validation_MSE=("Validation MSE", "mean"),
    Avg_Validation_MAE=("Validation MAE", "mean"),
    Avg_Validation_R2=("Validation R2", "mean"),
    Avg_Overfit_R2_Gap=("Overfit R2 Gap", "mean")
).reset_index()

display(regression_score_summary.round(4))

plt.figure(figsize=(10, 5))

sns.barplot(
    data=regression_score_summary,
    x="Classifier Source",
    y="Avg_Validation_R2",
    hue="Regression Model"
)

plt.title("So sánh R2 trung bình của hồi quy từ score phân loại")
plt.xlabel("Mô hình phân loại tạo score")
plt.ylabel("Average Validation R2")
plt.legend(title="Regression Model")
plt.show()

plt.figure(figsize=(10, 5))

sns.barplot(
    data=regression_score_summary,
    x="Classifier Source",
    y="Avg_Validation_MSE",
    hue="Regression Model"
)

plt.title("So sánh MSE trung bình của hồi quy từ score phân loại")
plt.xlabel("Mô hình phân loại tạo score")
plt.ylabel("Average Validation MSE")
plt.legend(title="Regression Model")
plt.show()

In [ ]:
best_regression_row = regression_score_results.sort_values(
    by=["Validation R2", "Validation MSE"],
    ascending=[False, True]
).iloc[0]

print("Mô hình hồi quy từ score tốt nhất:")
display(best_regression_row.to_frame().T.round(4))

In [ ]:
approach_comparison = all_classification_results.groupby(
    ["Model", "Group", "Type", "Data"]
).agg(
    Mean_Accuracy=("Validation Accuracy", "mean"),
    Mean_Balanced_Accuracy=("Validation Balanced Accuracy", "mean"),
    Mean_Precision=("Validation Precision", "mean"),
    Mean_Recall=("Validation Recall", "mean"),
    Mean_F1=("Validation F1-score", "mean"),
    Mean_AUC=("Validation AUC", "mean"),
    Mean_Overfit_Gap=("Overfit Gap", "mean")
).reset_index()

display(approach_comparison.round(4))

### **Tổng kết và so sánh các mô hình phân loại**

Kết quả trên tập validation stratified 60:40 với `random_state=42`:

| Mô hình | F1 | AUC | Recall |
|---|---:|---:|---:|
| Logistic Regression | 0.6227 | 0.8418 | 0.8142 |
| SVM RBF | 0.6204 | 0.8299 | 0.8008 |
| Naive Bayes | 0.6034 | 0.8137 | 0.8115 |

**Khuyến nghị:** Logistic Regression là mô hình phù hợp nhất để triển khai demo vì đạt F1, AUC và Recall tốt nhất trong ba mô hình, đồng thời nhẹ và dễ giải thích. SVM RBF và Naive Bayes vẫn hữu ích để so sánh cách tiếp cận phi tuyến và sinh.